This is one of solution notebooks of LB 8th.

Solution write up is here [https://www.kaggle.com/competitions/arc-prize-2025/writeups/lb10-00-with-small-change-to-2024architects#3313003](https://www.kaggle.com/competitions/arc-prize-2025/writeups/lb10-00-with-small-change-to-2024architects#3313003)

My final solution ended up being a lightly modified version of [the ARChitects2024approach](https://www.kaggle.com/code/dfranzen/arc-prize-2024-solution-by-the-architects). None of my other experiments managed to beat it, so huge thanks to the ARChitects team for making their work public.

Here are the main tweaks I made:

1 . Training

* TTT (test-time training) for each task
* num_epoch = 24
* Reduced num_train to 3 for faster training
* batch_size = 4
* grad_accumulation_step = 2
* Removed background lines for faster training in one of the two submissions (applied in this notebook)

2 . Inference

* n = 1
* Reduced num_train to 2 for faster inference
* Used 2 subprocesses to speed up inference
* top_k = 4 for DFS to reduce search time

With these hyperparameters, not all tasks finish within the 12-hour limit. I tried lowering num_epoch, but performance dropped a lot, so I stuck with 24.

**Background line removal**   
The background value is estimated by comparing the size of extracted objects to the size of the full grid. If a row or column contains only the background value in both the input and output grids, it becomes a candidate for removal.
When such background-only lines appear consecutively, the entire block is collapsed so that only one background line remains.
This preprocessing is applied only to the training grids; test grids are left unchanged.

* Version 1 is the submitted version.
* Version 3 is a cleaned-up version of V1 with some unnecessary lines removed.

In [ ]:
%%writefile model_runner.py
import json
import os, sys
import bz2
import pickle
import numpy as np24
from tqdm import tqdm
from pp import *
from visualize import *
import traceback

def indices_required_for_merges(keep_indices, vocab, merges):
    merges_lookup = {}
    for m in merges:
        a, b = m.split(' ') if isinstance(m, str) else m
        key = vocab[f'{a}{b}']
        if key not in merges_lookup: merges_lookup[key] = set()
        merges_lookup[key].add(vocab[a])
        merges_lookup[key].add(vocab[b])
    to_process = list(keep_indices)
    while len(to_process):
        for w in merges_lookup.get(to_process.pop(), []):
            if w not in keep_indices:
                keep_indices[w] = None
                to_process.append(w)
    return keep_indices

def remove_unused_merges(merges, vocab):
    return [f'{a} {b}' for a, b in [m.split(' ') if isinstance(m, str) else m for m in merges] if all(w in vocab for w in [a, b, a + b])]

def map_special_tokens(data, mapping=None):
    tokens = set()
    if isinstance(data, dict):
        special = data.get('special_tokens')
        if special is not None:
            for v in special.values():
                tokens.update(v['ids'])
                if mapping is not None:
                    v['ids'] = [mapping.get(i) for i in v['ids'] if i in mapping]
    for v in (data.values() if isinstance(data, dict) else data if isinstance(data, list) else []):
        tokens.update(map_special_tokens(v, mapping))
    return tokens

def remove_tokenizer_normalizer(tokenizer):
    from tokenizers import Tokenizer
    assert tokenizer.is_fast
    tokenizer_json = json.loads(tokenizer._tokenizer.to_str())
    if tokenizer_json.get('normalizer') is not None:
        tokenizer_json['normalizer'] = None
        tokenizer._tokenizer = Tokenizer.from_str(json.dumps(tokenizer_json))

def shrink_tokenizer_vocab(tokenizer, keep_indices, keep_special_tokens, keep_token_order):
    from tokenizers import Tokenizer
    assert tokenizer.is_fast
    tokenizer_json = json.loads(tokenizer._tokenizer.to_str())
    assert tokenizer_json['model']['type'] == "BPE"
    if keep_special_tokens:
        keep_indices.update({k: None for k in tokenizer.all_special_ids})
        keep_indices.update({k: None for k in map_special_tokens(tokenizer_json.get('post_processor'))})
    keep_indices = indices_required_for_merges(keep_indices, tokenizer_json['model']['vocab'], tokenizer_json['model']['merges'])
    if keep_token_order: keep_indices = sorted(keep_indices)
    mapping = {old: new for new, old in enumerate(keep_indices)}
    tokenizer_json['model']['vocab'] = {k: mapping[v] for k, v in tokenizer_json['model']['vocab'].items() if v in mapping}
    tokenizer_json['model']['merges'] = remove_unused_merges(tokenizer_json['model']['merges'], tokenizer_json['model']['vocab'])
    special_tokens_order = [t['id'] for t in tokenizer_json['added_tokens']]
    assert special_tokens_order==sorted(special_tokens_order)
    tokenizer_json['added_tokens'] = sorted([{**t, 'id': mapping[t['id']]} for t in tokenizer_json['added_tokens'] if t['id'] in mapping], key=lambda t: t['id'])
    map_special_tokens(tokenizer_json.get('post_processor'), mapping)
    tokenizer._tokenizer = Tokenizer.from_str(json.dumps(tokenizer_json))
    return mapping, keep_indices

def shrink_model_embeddings(model, keep_indices, mapping):
    import torch
    with torch.no_grad():
        row_select = torch.tensor(list(keep_indices))
        new_embed_t = torch.index_select(model.get_input_embeddings().weight.data, 0, row_select.to(model.get_input_embeddings().weight.data.device))
        new_lm_head = torch.index_select(model.get_output_embeddings().weight.data, 0, row_select.to(model.get_output_embeddings().weight.data.device))
        model.resize_token_embeddings(len(keep_indices))
        model.get_input_embeddings().weight.data[:] = new_embed_t
        model.get_output_embeddings().weight.data[:] = new_lm_head
        for config in [model.config, model.generation_config]:
            for k, v in list(config.to_dict().items()):
                if k.endswith('token_id'):
                    setattr(config, k, [mapping.get(t) for t in v] if isinstance(v, list) else mapping.get(v))

def shrink_embeddings(model, tokenizer, corpus=None, keep_token_ids=[], keep_tokens=[], remove_token_ids=[], keep_model_tokens=True, keep_special_tokens=True, keep_normalizer=False, keep_token_order=True):
    if not keep_normalizer: remove_tokenizer_normalizer(tokenizer)
    from collections import OrderedDict  # use as OrderedSet
    keep_indices = OrderedDict()
    keep_indices.update({k: None for k in keep_token_ids})
    keep_indices.update({tokenizer.vocab[t]: None for t in keep_tokens})
    if corpus is not None: keep_indices.update({k: None for k in tokenizer(corpus)['input_ids']})
    if keep_model_tokens:
        for config in [model.config, model.generation_config]:
            for k, v in config.to_dict().items():
                if k.endswith('token_id'):
                    keep_indices.update({k: None for k in (v if isinstance(v, list) else [v])})
    keep_indices.pop(None, None)
    for idx in remove_token_ids: keep_indices.pop(idx, None)
    mapping, keep_indices = shrink_tokenizer_vocab(tokenizer, keep_indices, keep_special_tokens, keep_token_order)
    shrink_model_embeddings(model, keep_indices, mapping=mapping)
    return mapping

def fix_dtypes(model, fix_weights=True, fix_quant_states=True):
    import torch
    for module in model.modules():
        weight = getattr(module, 'weight', None)
        if weight is not None:
            if torch.is_floating_point(weight):
                if fix_weights and weight.dtype!=model.dtype:
                    module.to(model.dtype)
            else:
                qs = getattr(weight, 'quant_state', None)
                if qs is not None:
                    if fix_quant_states and qs.dtype!=model.dtype:
                        qs.dtype = model.dtype
    return model

def merge_peft_into_base(model):
    print('*** Merge peft model into base model...')
    assert is_peft_model(model)
    return fix_dtypes(model.merge_and_unload())

def save_model(store_path, model=None, tokenizer=None, merge=False):
    if merge: model = merge_peft_into_base(model)
    if store_path is not None:
        assert model is not None or tokenizer is not None
        print(f"*** Saving{' merged' if merge else ''} model/tokenizer to '{store_path}'...")
        if model is not None: model.save_pretrained(store_path)
        if tokenizer is not None:
            tokenizer.save_pretrained(store_path)
            to_delete = os.path.join(store_path, 'tokenizer.model')
            if os.path.isfile(to_delete): os.remove(to_delete)
    return model

def is_unsloth_model(model):
    return model.model_tags is not None and 'unsloth' in model.model_tags

def is_peft_model(model):
    return hasattr(model, 'peft_type')

def download_model(repo_id, store_path, get_name=lambda n: os.path.join(n.replace('/', '--'), 'transformers', 'default', '1')):
    import os
    if os.path.exists(repo_id): return repo_id
    model_path = os.path.join(store_path, get_name(repo_id))
    if not os.path.exists(model_path):
        from huggingface_hub import snapshot_download
        download_path = snapshot_download(repo_id=repo_id)
        os.makedirs(os.path.split(model_path)[0], exist_ok=True)
        os.symlink(download_path, model_path, target_is_directory=True)
    return model_path

def get_and_fix_peft_weights(store):
    print(f"*** Load peft state_dict from '{store}'...")
    from peft import load_peft_weights
    state_dict = load_peft_weights(store)
    for k in list(state_dict.keys()):
        if 'modules_to_save' in k:
            del state_dict[k]
            original_module_key = k.replace('.modules_to_save.', '.original_module.')
            if original_module_key in state_dict: del state_dict[original_module_key]
            assert k.replace('.modules_to_save.', '.') in state_dict
    return state_dict

def set_peft_weights(model, state_dict):
    print(f"*** Set model state_dict...")
    from peft import set_peft_model_state_dict
    res = set_peft_model_state_dict(model, state_dict)
    assert not res.unexpected_keys

def load_peft_state(model, store):
    set_peft_weights(model, get_and_fix_peft_weights(store))

def prepare_model(model, mode, inf=False, tokenizer=None, formatter=None, shrink_embedding=False, dequantize=False, peft=[], local_files_only=False, add_special_tokens={}, set_pad_token=None, keep_tokens=[], keep_normalizer=None, peft_trainable=True, device_map=None, tf_grad_cp=True, tf_use_fa2=True, **kwargs):
    if isinstance(model, str):
        assert tokenizer is None
        print(f"*** Load base model and tokenizer from '{model}'...")
        if mode=='unsloth_4bit':
            assert device_map is None, 'unsupported'
            from unsloth import FastLanguageModel
            model, tokenizer = FastLanguageModel.from_pretrained(model_name=model, dtype=None, load_in_4bit=True, local_files_only=local_files_only, **kwargs)

        elif mode in ['transformers', 'transformers_bf16', 'transformers_4bit', 'transformers_bf16_4bit', 'tokenizer_only']:
            import torch
            model_load_args = {}
            if device_map is not None: model_load_args['device_map'] = device_map
            if tf_use_fa2: model_load_args['attn_implementation'] = 'flash_attention_2'
            if mode in ['transformers_bf16', 'transformers_bf16_4bit']: model_load_args['torch_dtype'] = torch.bfloat16
            if mode in ['transformers_4bit', 'transformers_bf16_4bit']:
                from transformers import BitsAndBytesConfig
                nf4_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
                model_load_args['quantization_config'] = nf4_config
            from transformers import AutoTokenizer, AutoModelForCausalLM
            tokenizer = AutoTokenizer.from_pretrained(model, local_files_only=local_files_only, **kwargs)
            model = AutoModelForCausalLM.from_pretrained(model, **model_load_args) if mode!='tokenizer_only' else None
            if tf_grad_cp and model is not None: model.gradient_checkpointing_enable()
        else: raise NotImplementedError('Unknown mode.')
    if add_special_tokens: tokenizer.add_special_tokens(add_special_tokens)
    if set_pad_token is not None: tokenizer.pad_token = set_pad_token
    if formatter is not None and not hasattr(formatter, 'corpus'):
        formatter = formatter(tokenizer=tokenizer)
    if (shrink_embedding<len(tokenizer.vocab) if type(shrink_embedding)==int else shrink_embedding) or keep_normalizer is False:
        print('*** Shrink embedding...')
        embedding_size_before_shrink = len(tokenizer.vocab)
        mapping = shrink_embeddings(model, tokenizer, formatter.get_corpus(), keep_tokens=keep_tokens, keep_normalizer=keep_normalizer)
        print(f'*** -> Reduced embedding size from {embedding_size_before_shrink} to {len(mapping)} words.')
    if dequantize:
        print(f'*** Dequantize model...')
        model = model.dequantize()

    if len(peft):
        peft_trained = True if is_peft_model(model) else None
        if pp:print('peft_trained',peft_trained)
        for i, m in enumerate(peft):
            if peft_trained is True: model, peft_trained = merge_peft_into_base(model), None
            if isinstance(m, str):
                if peft_trained is False:
                    _, peft_trained = load_peft_state(model, m), True
                else:
                    print(f"*** Load peft model from '{m}'...")
                    # be careful when using unsloth - using PeftModel to load the model will not apply unsloth optimizations
                    from peft import PeftModel
                    model, peft_trained = PeftModel.from_pretrained(model, m, trainable=peft_trainable), True
            else:
                assert peft_trained is None
                if isinstance(m, dict):
                    print('*** Create new peft model...')
                    if is_unsloth_model(model):
                        from unsloth import FastLanguageModel
                        my_get_peft_model = FastLanguageModel.get_peft_model
                    else:
                        from peft import LoraConfig, get_peft_model
                        my_get_peft_model = lambda model, **kwargs: get_peft_model(model, LoraConfig(**kwargs))
                    model, peft_trained = my_get_peft_model(model, **m), False
                else: assert m is None
    if inf:
        if is_unsloth_model(model):
            from unsloth import FastLanguageModel
            FastLanguageModel.for_inference(model)
        else: model.eval()
    return model, tokenizer, formatter

def training_run(model, formatter, dataset, train_args, max_seq_length, merge=False, store=None, packing=False, grad_acc_fix=False, optimizers=None):
    assert merge is False, "merge after training does not seen to work (at least with unsloth, saved merged model will cointain the untrained weights!)"
    import torch
    from datasets import Dataset
    add_train_args = {}
    if is_unsloth_model(model):
        from unsloth import FastLanguageModel
        from unsloth import UnslothTrainer as Trainer
        from unsloth import UnslothTrainingArguments as TrainingArguments
        from unsloth import is_bfloat16_supported
        FastLanguageModel.for_training(model)
        add_train_args.update(fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported())

    else:
        from trl import SFTConfig as TrainingArguments
        from trl import SFTTrainer as Trainer
        model.train()
        add_train_args.update(bf16=True)

    formatter.tokenizer.padding_side = 'right'
    if is_unsloth_model(model):
        for convert_to_float in [model.get_input_embeddings(), model.get_output_embeddings()]:
            if convert_to_float.weight.dtype!=torch.float32: convert_to_float.to(torch.float32)

    add_args = {}
    if optimizers is not None: add_args['optimizers'] = optimizers

    trainer = Trainer(
        model=model,
        tokenizer=formatter.tokenizer,
        data_collator=formatter.get_data_collator(),
        train_dataset=Dataset.from_list(dataset.as_list(formatter)),
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        dataset_num_proc=None,
        packing=packing,  # Can make training 5x faster for short sequences.
        **add_args,
        args=TrainingArguments(
            **add_train_args,
            **train_args
        ),
    )

    print('*** Start training run...')
    if (grad_acc_fix or train_args['gradient_accumulation_steps']==1) and is_unsloth_model(model):
        try:
            from unsloth import unsloth_train
            trainer_stats = unsloth_train(trainer)
        except:
            if pp:print(traceback.format_exc())
            trainer_stats = trainer.train()
    else:
        if is_unsloth_model(model) and train_args['gradient_accumulation_steps']>1: print('*** WARNING: using faulty unsloth gradient accumulation')
        trainer_stats = trainer.train()
    try: print(f'*** -> Training took {trainer_stats.metrics["train_runtime"]} seconds.')
    except: pass
    if store is not None: save_model(store, model, formatter.tokenizer, merge=merge)
    return model, trainer_stats, trainer

def inference_load(store, keys=True, result_dict=None, always_read_from_file=False):
    if pp:import traceback
    if result_dict is None: result_dict = {}
    if store is not None:
        if keys is True: keys = os.listdir(store)
        for key in keys:
            if always_read_from_file or key not in result_dict:
                try:
                    with bz2.BZ2File(os.path.join(store, key)) as f: result_dict[key] = pickle.load(f)
                except:
                    if pp:print(key,traceback.format_exc())
                    continue
    return result_dict

def inference_save(store, key, outputs):
    if store is not None:
        # if pp:print('inference_save, store:',store,'key:',key,'outputs:',[v[:-1] for v in outputs])
        os.makedirs(store, exist_ok=True)
        with bz2.BZ2File(os.path.join(store, key), 'w') as f: pickle.dump(outputs, f)

class Decoder(object):
    def __init__(self, formatter, dataset, n_guesses, cut_query=cut_query, rotp=rotp, max_outputs=None, frac_score=False, quiet=False, name='', additional_decoders=None, prob_baseline=None, keep_max=None):
        self.formatter = formatter
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}
        self.correct_solutions = {}
        self.keys_lim = set()
        self.keys_all = set()
        self.mult_cnt = {}
        self.keys_cnt = {}
        self.frac_score = frac_score
        self.max_outputs = max_outputs
        self.quiet = quiet
        self.input_len = [{} if formatter is not None and formatter.tokenizer is None else ds.get_lengths(formatter, name='input') for ds in [dataset, dataset.mod(np.transpose, keep_key=True)]]
        self.reply_len = [{} if formatter is not None and formatter.tokenizer is None else ds.get_lengths(formatter, name='reply') for ds in [dataset, dataset.mod(np.transpose, keep_key=True)]]
        self.additional_decoders = additional_decoders
        self.name = name
        self.prob_tracker = {}
        self.prob_tracker_best = {
            k: (0, dataset.queries[k]['test'][0]['input'])
            for k in dataset.keys
            if k in dataset.queries and all(
                np.array(v['input']).shape == np.array(v['output']).shape and
                np.sum(np.array(v['input']) != np.array(v['output'])) < (np.array(v['input']).size // 2)
                for v in dataset.queries[k]['train']
            )
        } if use_cb else {}
        self.prob_baseline = prob_baseline
        self.is_from_store = False
        self.keep_max = keep_max
        self.rotp = rotp
        self.cut_query_count = cut_query

    def score(self, *to_score):
        scores = [(sum(1/self.mult_cnt[k.split('_')[0]] for k in s) if self.frac_score else len(s)) for s in to_score]
        score_cnt = len(self.mult_cnt if self.frac_score else self.keys_cnt)
        return scores, score_cnt

    def from_store(self, store, **kwargs):
        self.is_from_store = True
        for key, outputs in tqdm(inference_load(store).items(), desc=f'inference_load from store {store}', file=sys.stdout) if pp else inference_load(store).items():
            if pp:print(key,[v[:2] for v in outputs])
            self.process(key, outputs, **kwargs)
        return self

    def score_fmt(self, v):
        return f'{v:5.1f}' if self.frac_score else f'{v:3}'

    def process_single_output(self, key, output_len, decoded, print_func=print, len_info=None, device_info=None, batch=False):
        import numpy as np
        import traceback
        inv_mod = {k: v if k.endswith('val') else self.dataset.invert_mod(v, key, inv_perm=(k.startswith('output') or k.startswith('score_all'))) for k, v in decoded.items()}
        base_key = key.split('.')[0]
        self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
        output = inv_mod.get('output')
        if base_key in self.dataset.queries and np.array_equal(np.array(output), np.array(self.dataset.queries.get(base_key)['test'][0]['input'])):
            if pp:print('Same as Input',key)
        else:
            self.decoded_results[base_key][key] = inv_mod
        if np.sum(output==-1):
            try:
                in_arr = self.dataset.queries.get(base_key)['test'][0]['input']
                in_arr = np.array(in_arr)
                output = np.array(output)  # ensure it's a NumPy array if not already
                
                mask = (output == -1)
                output[mask] = in_arr[mask]
            except:
                if pp:print(traceback.format_exc())
        score = inv_mod.get('score')

        # quick scoring
        self.keys_cnt[base_key] = self.keys_cnt.get(base_key, 0) + 1
        mult_key, mult_sub = (base_key.split('_') + ['0'])[:2]
        self.mult_cnt[mult_key] = max(self.mult_cnt.get(mult_key, 0), int(mult_sub) + 1)
        if len(self.dataset.replies):
            correct_solution = self.dataset.replies.get(base_key)
            if correct_solution is None:
                correct_solution = self.dataset.queries.get(base_key.split('_')[0]+'_0')
                if correct_solution is not None:
                    try:
                        correct_solution = correct_solution['train'][9-int(base_key.split('_')[1])]['output']
                        correct_solution = [correct_solution]
                    except:
                        pass
            if correct_solution is not None:
                correct_solution = correct_solution[0]
                self.correct_solutions[base_key] = correct_solution
                is_correct = correct_solution is not None and np.array_equal(correct_solution, output)
                if is_correct:
                    self.keys_all.add(base_key)
                    if self.keys_cnt[base_key] <= self.n_guesses: self.keys_lim.add(base_key)

            corr_str = 'cant_decode' if output is None else 'sol_unknown' if correct_solution is None else 'ALL_CORRECT' if is_correct else 'bad_xy_size' if np.shape(correct_solution)!=np.shape(output) else 'bad_content'
            (score_lim, score_all), score_cnt = self.score(self.keys_lim, self.keys_all)

        if output is not None and (use_cb or pp) and self.is_from_store:
            tp_arr = (key.count('transpose') + key.count('rot90')) % 2
            msc = None if score is None else np.sum(score)
            fsc = inv_mod.get('score_val')
            pt = self.prob_tracker[base_key] = self.prob_tracker.get(base_key, {})
            hash = tuple(map(tuple, output))
            # if pp:
            #     if self.prob_baseline is None:
            #         print(f'prob=pt.get(hash, 0){pt.get(hash, 0):.1f} + (np.exp(fsc{fsc:.1f})){np.exp(fsc):.1f}')
            #     else:
            #         print(f'prob=pt.get(hash, 0){pt.get(hash, 0):.1f} + fsc{fsc:.1f} - np.log(self.prob_baseline{self.prob_baseline:.1f}){np.log(self.prob_baseline):.1f}')
            pt_get = pt.get(hash,0)
            prob = pt[hash] = pt.get(hash, 0) + (np.exp(fsc) if self.prob_baseline is None else fsc - np.log(self.prob_baseline))
            # if pp:print('prob=',prob)
            if batch and os.path.exists(f'current_best_{base_key}'):
                try:
                    with open(f'current_best_{base_key}', 'rb') as f:
                        current_best = pickle.load(f)
                except:
                    if pp:print(f'tried current_best_{base_key} open',traceback.format_exc())
                    current_best = self.prob_tracker_best.get(base_key)
            else:
                current_best = self.prob_tracker_best.get(base_key)
            if current_best is None or current_best[0]<prob:
                if batch:
                    with open(f'current_best_{base_key}', 'wb') as f:
                        pickle.dump((prob, output),f)
                    if pp:print(f'\nupdated current_best_{base_key} in pickle',round(prob,2))
                # else:
                self.prob_tracker_best[base_key] = (prob, output)
                # if pp:print(f'updated current_best_{base_key} in self',(round(prob,2), output))
                # if current_best is not None:
                #     if pp:print('prob_tracker_best',round(current_best[0],3),'->',round(prob,3),key)
            if pp:
                fmt_name = f'{self.name}: ' if self.name else ''
                # msc_print = f'{min(-msc, 9.99999):7.5f}' if msc is not None else 'unknown'
                # fsc_print = f'{min(-fsc, 9.99999):7.5f}' if fsc is not None else 'unknown'
                msc_print = f'{-msc:.3f}' if msc is not None else 'unknown'
                fsc_print = f'{-fsc:.3f}' if fsc is not None else 'unknown'
                if not self.quiet: print_func(f" {fmt_name}acc: {self.score_fmt(score_lim)}/{score_cnt:3}={min(score_lim/score_cnt, 0.999):5.1%} (2-guess), {self.score_fmt(score_all)}/{score_cnt:3}={min(score_all/score_cnt, 0.999):5.1%} (any);{f' {device_info}' if device_info else ''} tok:{self.input_len[tp_arr].get(base_key, '?'):>4}+{self.reply_len[tp_arr].get(base_key, '?'):>3}>{'n/a' if output_len is None else output_len:>3} {corr_str}:{msc_print}|{fsc_print} [{key}]")
                if correct_solution is not None and output is not None and self.is_from_store:
                    visualize_task(output, title=f"{key},msc:{msc_print},fsc:{fsc_print},prob:{prob:.5f}(pt{pt_get}) {corr_str}", answer=correct_solution, prob_grid=score)

    def get_current_best(self, base_key, batch=False):
        if batch and os.path.exists(f'current_best_{base_key}'):
            try:
                with open(f'current_best_{base_key}', 'rb') as f:
                    current_best = pickle.load(f)
            except:
                if pp:print(f'tried current_best_{base_key} open',traceback.format_exc())
                current_best = self.prob_tracker_best.get(base_key)
        else:
            current_best = self.prob_tracker_best.get(base_key)
        return None if current_best is None else current_best[1]

    def process_single_decode(self, key, de_tokenized, print_func=print, **kwargs):
        if len(de_tokenized)==3 and not isinstance(de_tokenized[1], float):  # for backwards compatibility
            output_len, *data = de_tokenized
            score_val = None
        else: output_len, score_val, *data = de_tokenized
        if self.formatter is None:
            assert len(data) == 1
            decoded = [data[0]]
        else: decoded = self.formatter.decode_to_array(*data)
        for d in decoded: d['score_val'] = score_val
        for i, dec in enumerate(decoded):
            if i==0: self.process_single_output(key, output_len, dec, print_func=print_func, **kwargs)
            elif self.additional_decoders:
                if i-1<len(self.additional_decoders): self.additional_decoders[i-1].process_single_output(key, output_len, dec, print_func=print_func, **kwargs)
                else: print_func(f'{key} no decoder available for output #{i}')
            else: self.process_single_output(f'{key}.fix{i}', output_len, dec, print_func=print_func, **kwargs)

    def process(self, key, de_tokenized, **kwargs):
        for i, d in enumerate(de_tokenized):
            if self.max_outputs is None or i<=self.max_outputs:
                self.process_single_decode(f'{key}.out{i}', d, **kwargs)

    def get_unsolved_keys(self):
        unsolved = []
        for base_key, reply in self.dataset.replies.items():
            if not any(np.array_equal(reply[0], s.get('output')) for s in self.decoded_results.get(base_key, {}).values()):
                unsolved.append(base_key)
        return unsolved

    def run_selection_algo(self, selection_algorithm):
        return {bk: (selection_algorithm({k: g for k, g in v.items() if g.get('output') is not None}) if any(g.get('output') is not None for g in v.values()) else []) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self, selection_algorithms, skip_failed=True):
        import numpy as np
        results = {}
        print('*** Benchmark selection algorithms...')
        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            try:
                selected = self.run_selection_algo(selection_algorithm)
                if self.formatter is not None:
                    for sols in selected.values():
                        for s in sols:
                            assert self.formatter.is_valid_solution(s), f'found invalid solutions {s}'
                correct_keys = {k for k, v in selected.items() if self.correct_solutions.get(k) is not None and any(np.array_equal(guess, self.correct_solutions[k]) for guess in v[:self.n_guesses])}
                (score,), score_cnt = self.score(correct_keys)
                results[name] = score
                print(f" acc: {score:5.1f}/{score_cnt:3}={score/score_cnt:6.2%} ('{name}')")
            except:
                print(f" {'execution failed':>21} ('{name}')")
                if not skip_failed: raise
        return results

    def calc_augmented_scores(self, model, base_keys=None, store=None, seed=0, max_len=None, make_unique=False, quiet=False, **kwargs):
        if base_keys is None: base_keys = list(self.decoded_results.keys())
        if store is not None: store = f'{store}_new'  # new format is not backwards compatible, so use new folder
        for bk in (base_keys if quiet else tqdm(base_keys, desc=f'calculate augmented scores for base_keys {base_keys}', file=sys.stdout)):
            res = self.decoded_results.get(bk, {})
            known_scores = {}
            for k, v in sorted(res.items()) if quiet else tqdm(sorted(res.items()), desc=f'calc aug bk_{bk}', file=sys.stdout):
                # if pp:print('calc_augmented_scores',k,'score_val',v.get('score_val'))
                if 'output' in v:
                    k_store = None if store is None else os.path.join(store, k)
                    id = tuple(map(tuple, v['output']))

                    # if pp:make_unique=False
                        
                    if not (make_unique and id in known_scores):
                        try:
                            assert k_store is not None
                            with bz2.BZ2File(k_store) as f: known_scores[id] = pickle.load(f)
                            if isinstance(known_scores[id], list): known_scores[id] = dict(score_multi=known_scores[id])  # for backwards compatibility
                            # if pp:print('with bz2file',k)
                            k_store = None
                        except Exception as e:
                            if model is None:
                                if pp:print(f"[Warning] Failed to load {k_store}: {e}")
                                continue
                            temp_dataset = self.dataset.__class__(
                                keys=[bk],
                                queries={bk: self.dataset.queries.get(bk)},
                                replies={bk: [v['output'].tolist()]},
                            )
                            temp_decoder = self.__class__(self.formatter, temp_dataset, n_guesses=self.n_guesses, quiet=True)
                            temp_dataset = temp_dataset.augment(**kwargs, seed=(seed+hash(k)+hash(id)) % 1024**2, quiet=True, keep_max=self.keep_max)
                            temp_dataset = temp_dataset.cut_to_query_count(self.cut_query_count)
                            if self.cut_query_count>2:temp_dataset = temp_dataset.cut_to_query_count(self.cut_query_count-1,p=0.5)
                            if self.rotp:
                                temp_dataset = temp_dataset.one_rotp()
                            if max_len is not None: temp_dataset = temp_dataset.cut_to_len(formatter=self.formatter, name='input', max_len=max_len, quiet=True)
                            # if not quiet: pbar = tqdm(temp_dataset.as_list(self.formatter), desc=f'calc_score {bk}',file=sys.stdout)
                            for x in temp_dataset.as_list(self.formatter): calc_score(**x, formatter=self.formatter, model=model, decoder=temp_decoder)
                            if pp and id in known_scores:
                                print('Previous known_scores',known_scores[id]['score_multi_nl'],sum(known_scores[id]['score_multi_nl']))
                                print(k,[x['score_val'] for x in temp_decoder.decoded_results[bk].values()],sum([x['score_val'] for x in temp_decoder.decoded_results[bk].values()]))
                            known_scores[id] = dict(
                                score_multi=[np.sum(x['score']) for x in temp_decoder.decoded_results[bk].values()],
                                score_multi_nl=[x['score_val'] for x in temp_decoder.decoded_results[bk].values()],
                                score_multi_array=np.array([x['score'] for x in temp_decoder.decoded_results[bk].values()]),
                                score_multi_array_cum=np.array([x['score_cum'] for x in temp_decoder.decoded_results[bk].values()]),
                                score_multi_array_all=np.array([x['score_all'] for x in temp_decoder.decoded_results[bk].values()]),
                                score_multi_array_all_cum=np.array([x['score_all_cum'] for x in temp_decoder.decoded_results[bk].values()]),
                            )
                            # if pp:print(k,id,'score_val',known_scores[id]['score_multi_nl'])
                            if k_store is not None:
                                os.makedirs(store, exist_ok=True)
                                with bz2.BZ2File(k_store, 'w') as f: pickle.dump(known_scores[id], f)
                    # if 'score_multi_nl' in v and pp:
                    #     print('\n\nV. BEFORE UPDATE',k,v.get('score_multi_nl'),'sum',sum(v.get('score_multi_nl')))
                    v.update(known_scores[id])

from collections import namedtuple

InferencePath = namedtuple("InferencePath", ["score", "tokens", "logits"])

def turbo_dfs(
    model, logits, path, eos_token_id, max_new_tokens, max_score, max_score_greedy,
    temperature, suppress_tokens, torch, score=0.0, pos=0, cache=None, top_k=None
):
    logits, next_logits = logits[0], (logits[1:] if len(logits) > 1 else None)
    nll = -(logits / temperature).detach().float().log_softmax(-1).cpu().numpy()
    greedy_index = nll.argmin(-1).item()
    
    # Build prioritized token list
    nll = list(enumerate(nll))
    if path:
        nll.sort(key=lambda x: (x[0] != path[0], x[1]))  # path[0] first, then lowest score
        path = path[1:]
    else:
        nll.sort(key=lambda x: x[1])  # sort by score

    if top_k:
        nll = nll[:top_k]

    results = []
    logits_cache = {}

    for i, s in nll:
        if i in suppress_tokens:
            continue

        next_score = score + s
        allowed_score = max_score_greedy if i == greedy_index else max_score

        if next_score >= allowed_score:
            continue

        if i == eos_token_id:
            results.append(InferencePath(next_score, [i], [logits]))
            continue

        if max_new_tokens <= 1:
            continue

        if i not in logits_cache:
            with torch.no_grad():
                input_ids = torch.full((1, 1), i, device=model.device)
                position_ids = torch.full((1, 1), pos, device=model.device)
                next_logits_, cache[0] = model(
                    input_ids=input_ids,
                    position_ids=position_ids,
                    past_key_values=cache[0],
                )[:2]
                logits_cache[i] = next_logits_[0]

        child_suffixes = turbo_dfs(
            model,
            logits=logits_cache[i],
            path=path,
            eos_token_id=eos_token_id,
            max_new_tokens=max_new_tokens - 1,
            max_score=max_score,
            max_score_greedy=allowed_score,
            temperature=temperature,
            suppress_tokens=suppress_tokens,
            torch=torch,
            score=next_score,
            pos=pos + 1,
            cache=cache,
            top_k=top_k
        )

        for child in child_suffixes:
            results.append(InferencePath(child.score, [i] + child.tokens, [logits] + child.logits))

    return results

def inference_turbo_dfs(
    model, input_ids, eos_token_id, max_new_tokens, min_prob,
    min_prob_greedy=1.0, temperature=0.9, suppress_tokens=[],
    path=[], attention_mask=None, top_k=4
):
    import torch
    with torch.no_grad():
        assert attention_mask is None or attention_mask.all(), 'attention_mask not fully supported'
        input_ids = torch.as_tensor(input_ids, device=model.device)
        if input_ids.ndim == 2:
            input_ids = input_ids.squeeze(0)
        assert input_ids.ndim == 1, 'batching not supported'

        max_score = -np.log(min_prob)
        max_score_greedy = -np.log(min_prob_greedy) if min_prob_greedy > 0 else float('inf')
        max_score_greedy = max(max_score, max_score_greedy)

        if path and path[-1] == eos_token_id:
            path = path[:-1]

        full_path = input_ids
        if path:
            path_tensor = torch.as_tensor(path, device=model.device)
            full_path = torch.cat([full_path, path_tensor])

        logits, cache = model(input_ids=full_path[None])[:2]
        logits = logits[0, len(input_ids) - 1:]

        paths = turbo_dfs(
            model, logits, path, eos_token_id, max_new_tokens,
            max_score, max_score_greedy, temperature,
            suppress_tokens, torch, score=0.0, pos=len(input_ids),
            cache=[cache], top_k=top_k
        )

        return sorted([
            (p.score, np.array(p.tokens), torch.stack(p.logits).float().cpu().numpy())
            for p in paths
        ], key=lambda x: x[0])

def inference_step(tokenized, model, text_g=None, shape=None, classifier=None, remove_token_type_ids=True, num_beams=1, formatter=None, min_prob=None, current_best=None, **kwargs):
    import torch
    if remove_token_type_ids: tokenized.pop('token_type_ids', None)
    if min_prob is not None:
        assert num_beams==1
        gen = inference_turbo_dfs(model, **tokenized.to(model.device), path=current_best, min_prob=min_prob, eos_token_id=formatter.tokenizer.eos_token_id, **kwargs)
        torch.cuda.empty_cache()

        tokens_out = [[g[1] for g in gen]]
        scores_out = [[g[2] for g in gen]]

    elif is_unsloth_model(model) and num_beams > 1:
        assert False, 'unsloth does not support beam search'

    else:
        gen = model.generate(**tokenized.to(model.device), return_dict_in_generate=True, output_logits=True, use_cache=True, **kwargs)
        tokens_out = gen['sequences'][:, torch.newaxis, tokenized['input_ids'].shape[-1]:].cpu().numpy().copy()
        scores_out = torch.stack(gen['logits'], axis=-2)[:, torch.newaxis].float().cpu().numpy().copy()
    return tokens_out, scores_out

def process_inference_output(key, outputs, formatter, classifier=None, store=None, decoder=None, decoder_args={}):
    de_tokenized = [formatter.de_tokenize(*output, classifier=classifier) for output in zip(*outputs)]
    #from detokenize: return max(len(tokens)+1, len(tokens_cut)), score_val, de_tokenized, scores
    inference_save(store, key, de_tokenized)
    if decoder is not None: decoder.process(key, de_tokenized, **decoder_args)
    return de_tokenized
    
def inference_run_v2(model, formatter, dataset, decoder=None, classifier=None, max_new_tokens=None, max_batch_size=1, store=None, result_dict=None, rerun_empty=False, retrain=None, use_turbo=False, group_multi_output=True, **kwargs):
    import torch
    assert max_batch_size==1, 'unsupported'

    with torch.no_grad():
        print('*** Load stored data...')
        if result_dict is None: result_dict = {}
        # result_dict = inference_load(store, dataset.keys, result_dict)
        by_base_key = {}
        needs_rerun = {}
        base_key_list = []
        for key in dataset.keys:
            base_key = key.split('.')[0]
            if group_multi_output: base_key = base_key.split('_')[0]
            if base_key not in by_base_key: base_key_list.append(base_key)
            bk_list = by_base_key[base_key] = by_base_key.get(base_key, [])
            bk_list.append(key)
        for base_key, keys in by_base_key.items():
            for key in keys:
                de_tokenized = result_dict.get(key)
                if de_tokenized is None or (rerun_empty and not de_tokenized):
                    bk_list = needs_rerun[base_key] = needs_rerun.get(base_key, [])
                    bk_list.append(key)
                elif decoder is not None: decoder.process(key, de_tokenized)

        formatter.tokenizer.padding_side = 'left'
        if max_new_tokens is None: max_new_tokens = formatter.max_new_tokens()
        if is_unsloth_model(model):
            from unsloth import FastLanguageModel
            FastLanguageModel.for_inference(model)
        else: model.eval()

        if pp:print('*** Start inference run...with max_new_tokens',max_new_tokens, 'with kwargs',kwargs)
    try:
        with tqdm(base_key_list, file=sys.stdout) as pbar:
            for base_key in pbar:
                run_keys = needs_rerun.get(base_key)
                if run_keys:
                    if retrain is not None:
                        retrain_dataset = dataset.keep_key_startswith(base_key)
                        print(f"retraining model for key '{base_key}' (retrain_dataset_size={len(retrain_dataset.keys)})")
                        retrain(model, retrain_dataset)
                        if is_unsloth_model(model): FastLanguageModel.for_inference(model)
                    with torch.no_grad():
                        with tqdm(run_keys, file=sys.stdout, desc=f"{base_key}") as pbarr:
                            for key in pbarr:
                                if pp:print(key, '\n')
                                input_text = dataset.get(key, formatter)['input']
                                if classifier is not None: 
                                    if det_size:
                                        last_input = input_text.rsplit('I',1)[-1].split('+')[0].strip()
                                        last_input_len = len(last_input.replace('\n',''))
                                        last_input_grid = last_input.splitlines()
                                        last_input_shape = (len(last_input_grid),len(last_input_grid[0]))
                                        text_g = input_text.replace('g','G').replace(last_input,''.join('g' if v.isdigit() else v for v in last_input))
                                        # input_text = input_text.replace('g','').replace('p','') + 'g'*last_input_len
                                        if pp:
                                            import random
                                            if random.randint(0,500)==0:print('input_text',input_text)
                                    else:
                                        input_text = input_text.replace('g','').replace('p','') + 'g'*900
                                batch = formatter.tokenizer([input_text], return_tensors='pt')
                                batch_g = formatter.tokenizer([text_g], return_tensors='pt') if det_size else None
                                # if pp:print('batch after tokenize',batch)
                                if classifier is not None:
                                    use_turbo=False
                                    kwargs['min_prob'] = None
                                current_best = decoder.get_current_best(key.split('.')[0]) if use_turbo else None
                                if current_best is not None:
                                    current_best = dataset.forward_mod(current_best, key)
                                    current_best = formatter.fmt_reply([current_best])
                                    current_best = formatter.tokenizer(input_text+current_best)['input_ids'][batch['input_ids'].shape[-1]:]
                                    # if pp:print('current_best_t',current_best)
                                try:
                                    import traceback
                                    batch_out = inference_step(batch, model, text_g=batch_g, shape=last_input_shape if det_size else None, classifier=classifier, formatter=formatter, max_new_tokens=max_new_tokens, current_best=current_best, **kwargs)
                                    outputs = [x[0] for x in batch_out]
                                    # if pp:print('outputs before process_inference_output',outputs)
                                    result_dict[key] = process_inference_output(key, outputs, formatter, classifier=classifier, store=store, decoder=decoder, decoder_args=dict(print_func=pbar.write))
                                except:
                                    print('Failed in inference_step or process_inference_output',key,traceback.format_exc())
                                    continue
        print('*** Completed inference run.')
    except KeyboardInterrupt: print('*** Ctrl+C pressed, stopping inference run.')
    return result_dict
    
# === inference_run_batch ===
def inference_run_batch(dataset, storage_path, gpu=0, max_new_tokens=None, store=None, result_dict=None, min_prob=0.1, min_prob_greedy=1.0, temperature=0.9, max_subprocesses=2, use_turbo=True, **kwargs):
    """Launch subprocess workers per base_key, limited to max_subprocesses in parallel. Runs 2 subprocesses even if only 1 base_key."""
    from collections import defaultdict
    import sys, subprocess, tempfile, os, pickle

    dataset_keys = dataset.keys

    # Group keys by base_key
    by_base_key = defaultdict(list)
    for key in dataset_keys:
        base_key = key.split('.')[0]
        by_base_key[base_key].append(key)

    base_keys = list(by_base_key.keys())

    key_separated = len(base_keys)==1
    # Ensure at least 2 base_keys for subprocesses
    if key_separated:
        base_keys = base_keys * max_subprocesses  # Duplicate to run 2-3 subprocesses

    result_dict = {} if result_dict is None else result_dict
    next_index = 0
    running = []

    def launch(base_key, index):
        out_path = tempfile.NamedTemporaryFile(delete=False).name
        cmd = [
            sys.executable, "subprocess_worker.py",
            base_key, storage_path, str(gpu), out_path,
            str(max_new_tokens), str(min_prob), str(min_prob_greedy), str(temperature), str(index), str(store), str(int(key_separated))
        ]
        p = subprocess.Popen(cmd)
        return (p, base_key, out_path)

    if pp:print('*** Start inference batch...with max_new_tokens',max_new_tokens, 'min_prob',min_prob,'max_subp',max_subprocesses, 'kwargs',kwargs)
        
    while next_index < len(base_keys) or running:
        # Launch new processes if slots available
        while len(running) < max_subprocesses and next_index < len(base_keys):
            base_key = base_keys[next_index]
            running.append(launch(base_key, next_index))
            next_index += 1

        # Wait for any process to finish
        for i, (p, base_key, out_path) in enumerate(running):
            if p.poll() is not None:
                if p.returncode != 0:
                    raise RuntimeError(f"Subprocess for {base_key} failed.")
                with open(out_path, 'rb') as f:
                    sub_result = pickle.load(f)
                os.remove(out_path)
                result_dict.update(sub_result)
                running.pop(i)
                break

    if key_separated and os.path.exists(f'current_best_{base_keys[0]}'):
        os.remove(f'current_best_{base_keys[0]}')
    # Combine memory stats if present
    mem_values = [v for k, v in result_dict.items() if 'mem' in k]
    if mem_values:
        result_dict['mem'] = max(mem_values)

    print("*** Completed all subprocesses.")
    return result_dict



class Retrainer(object):
    def __init__(self, n, aug_opts, reload_state_dict=None, **kwargs):
        self.n = n
        self.aug_opts = aug_opts
        self.reload_state_dict = reload_state_dict
        self.kwargs = kwargs

    def preprocess(self, dataset):
        ds = [dataset.augment(quiet=True, shfl_keys=True, **self.aug_opts) for _ in range((self.n-1)//dataset.length()+1)]
        ds = ds[0] if len(ds)==1 else ds[0].append(*ds[1:])
        ds, _ = ds.split_at_pos(self.n)
        return ds

    def __call__(self, model, dataset):
        if self.reload_state_dict is not None: set_peft_weights(model, self.reload_state_dict)
        assert is_unsloth_model(model), 'not implemented'
        if is_unsloth_model(model):
            from unsloth import FastLanguageModel
            FastLanguageModel.for_training(model)
        else: model.train()
        training_run(model, dataset=self.preprocess(dataset), **self.kwargs)

def calc_score(key, input, reply, formatter, model, store=None, decoder=None, **_):
    import torch
    # if pp:print('key in calc_score',key)
    if pp:
        import random
        if random.random()<0.001:
            print(key,'calc_score input',input,'calc_score reply',reply)
    with torch.no_grad():
        input_len = len(formatter.tokenizer(input)['input_ids'])
        tokenized = formatter.tokenizer([input+reply], return_tensors='pt')
        reply_tok = tokenized['input_ids'][0][input_len:].cpu().numpy().copy()
        reply_log = model.forward(**tokenized.to(model.device))['logits'][0, input_len-1: -1].float().cpu().numpy().copy()
        # if pp:print('reply_log',reply_log)
        process_inference_output(key, (reply_tok[torch.newaxis], reply_log[torch.newaxis]), formatter, store=store, decoder=decoder)

def mem_info(text='',gpu_id=0,path=None):
    import torch, traceback
    try:
        # if pp:print(torch.cuda.memory_summary())

        gpu_stats = torch.cuda.get_device_properties(gpu_id)
        usage = torch.cuda.max_memory_reserved() / 1024**3
        avail = gpu_stats.total_memory / 1024**3
        torch.cuda.empty_cache()  # Frees up unused cached memory (optional)
        torch.cuda.reset_peak_memory_stats()
        if pp:print(f"***{text} GPU: {gpu_stats.name}, used {usage:.3} / {avail:.3} GB.")
        return usage, [0,0]
    except: print(f'***{text} Exception occured when getting memory stats.',traceback.format_exc())

def disk_info(path):
    import shutil
    import os
    
    # Get current directory
    current_dir = os.getcwd()
    tmp_dir = '/kaggle'
    dirs = [current_dir,tmp_dir]

    u=[]

    for _dir in dirs:
        # Get disk usage for the mount point of current directory
        total, used, free = shutil.disk_usage(_dir)
        if pp:
            print(f" directory: {_dir}")
            print(f"Total: {total / (1024**3):.2f} GB")
            print(f"Used: {used / (1024**3):.2f} GB")
            print(f"Free: {free / (1024**3):.2f} GB")
        u.append(used/(1024**3)-6262 if used/(1024**3)>1000 else used/(1024**3))
    return u


In [ ]:
%%writefile subprocess_worker.py
# === subprocess_worker.py ===
import time
import_timer = time.time()
import sys
import torch
import pickle
from common_stuff import prepare_dataset, Decoder, arc_test_set, prepare_run, aug_score_params, score_temp_storage, keep_max
from model_runner import inference_step, process_inference_output, mem_info, disk_info
from pp import *
from tqdm import tqdm
if pp:print('import time',round(time.time()-import_timer))

if __name__ == "__main__":
    base_key, storage_path, gpu, out_path, max_new_tokens, min_prob, min_prob_greedy, temperature, index, store, key_separated = sys.argv[1:]

    gpu = int(gpu)
    # max_new_tokens = int(max_new_tokens)
    min_prob = float(min_prob)
    min_prob_greedy = float(min_prob_greedy)
    temperature = float(temperature)
    i = int(index)
    if store == 'None':
        store = None
    key_separated = bool(int(key_separated))

    if pp:mem_info(f'{i}.before_model_'+base_key)
    start_timer = time.time()
    model, formatter = prepare_run(storage_path, gpu=gpu, inf=True)
    prepare_run_fin = time.time()
    prepare_run_time = prepare_run_fin-start_timer
    if pp:
        print(i,'prepare_run_time',round(prepare_run_time))
        mem_info(f'{i}.after_model_'+base_key)
    if max_new_tokens == "None":
        max_new_tokens = formatter.max_new_tokens()
    else:
        max_new_tokens = int(max_new_tokens)

    if pp:print('max_new_tokens',max_new_tokens)

    dataset = prepare_dataset(formatter, train=False, gpu=gpu, key=base_key.split('_')[0])
    base_keys = []
    for key in dataset.keys:
        if key.startswith(base_key):
            base_keys.append(key)

    if len(base_keys)==len(dataset.keys) and key_separated:
        # base_keys = base_keys[::-i]
        base_keys = base_keys[i::2]
    if pp:print(i,'base_keys len:',len(base_keys),base_keys)
    decoder = Decoder(formatter, arc_test_set.keep_key_startswith(base_key.split('_')[0]).split_multi_replies(), n_guesses=2, prob_baseline=0.05, keep_max=keep_max)
    # if pp:print('probtracker_best',decoder.prob_tracker_best)
    # if pp:mem_info(f'{i}.after_decoder_'+base_key)

    formatter.tokenizer.padding_side = 'left'
    result_dict = {}

    data_decoder_fin = time.time()

    cb_time = 0
    for key in tqdm(base_keys, file=sys.stdout, desc=f'sp_inf{i} {base_key}') if pp else base_keys:
        # if pp:print(i,'\n')
        input_text = dataset.get(key, formatter)['input']
        batch = formatter.tokenizer([input_text], return_tensors='pt')

        cb_start = time.time()
        current_best = decoder.get_current_best(key.split('.')[0],batch=key_separated)
        cb_time += time.time()-cb_start
        if current_best is not None:
            # if pp:print(i,'current_best',current_best[0])
            current_best = dataset.forward_mod(current_best, key)
            current_best = formatter.fmt_reply([current_best])
            current_best = formatter.tokenizer(input_text + current_best)['input_ids'][batch['input_ids'].shape[-1]:]

        batch_out = inference_step(
            batch, model, formatter=formatter, max_new_tokens=max_new_tokens,
            current_best=current_best, min_prob=min_prob, min_prob_greedy=min_prob_greedy, temperature=temperature
        )
        outputs = [x[0] for x in batch_out]

        # if pp:print('\noutputs',outputs)
        result_dict[key] = process_inference_output(
            key, outputs, formatter, decoder=decoder, store=store, decoder_args={'batch':key_separated}
        )

    inf_fin = time.time()
    inf_time = inf_fin - data_decoder_fin
    if pp:print(i,'inf_time',round(inf_time),'key_separated',key_separated,'cb_time',cb_time)

    if use_aug:decoder.calc_augmented_scores(model=model, store=score_temp_storage, quiet=not pp, **aug_score_params)
    calc_fin = time.time()
    calc_time = calc_fin - inf_fin
    if pp:print(i,'calc_time',round(calc_time))

    result_dict[f'mem{i}'],_ = mem_info(f'{i}.subprocess_completed_'+base_key)
    with open(out_path, 'wb') as f:
        pickle.dump(result_dict, f)

    print(f"*** Subprocess for {base_key} completed.",round(time.time()-start_timer),'s')



In [ ]:
%%writefile visualize.py
import json
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import colors
import traceback

ARC_COLORMAP = colors.ListedColormap(
    ['#000000', '#0074D9', '#FF4136', '#2ECC40', '#FFDC00',
     '#AAAAAA', '#F012BE', '#FF851B', '#7FDBFF', '#870C25', '#FFFFFF']
)
ARC_NORM = colors.Normalize(vmin=0, vmax=10)

def visualize_task(
    task_data,
    task_solutions=None,
    n_test=0,
    title="ARC Task",
    figsize=(6, 3),
    save=False,
    grid=True,
    answer=None,
    file='eval',
    prob_grid=None,
    key=None,
    score=None,
):
    # if competition_run:
    #     return

    if answer is None:
        answer = []

    try:
        if file == 'eval':
            file = "/kaggle/input/arc-prize-2025/arc-agi_evaluation_challenges.json"
        # elif file == 'train':
        #     file = file.replace('evaluation','training')
        with open("/kaggle/input/arc-prize-2025/arc-agi_test_challenges.json" if not file else file, 'r') as f:
            test = json.load(f)
            
        with open("/kaggle/input/arc-prize-2025/arc-agi_training_challenges.json", 'r') as f:
            test.update(json.load(f))

        if score is not None and key:
            if file:
                sol_file = file.replace('test','training').replace('challenges','solutions')
            with open("/kaggle/input/arc-prize-2025/arc-agi_training_solutions.json" if not file else sol_file, 'r') as f:
                answer = json.load(f).get(key.split('_')[0],[])
                ind = int(key.split('_')[-1])
                if answer:
                    try:
                        answer = answer[ind]
                    except:
                        answer = test.get(key.split('_')[0])['train'][9-ind]['output']
            
        if isinstance(task_data,str):
            task_id = task_data
            title = task_id + ' ' + title
            task_data = test.get(task_data,{})
    
            if file:
                sol_file = file.replace('test','training').replace('challenges','solutions')
            with open("/kaggle/input/arc-prize-2025/arc-agi_training_solutions.json" if not file else sol_file, 'r') as f:
                solutions = json.load(f).get(task_id,[])
            if task_solutions is None:
                task_solutions = solutions
            else:
                answer = solutions
                
    except:
        print('Load in vis failed.')
        try:
            print(traceback.format_exc())
        except:
            pass
        return

    if prob_grid is not None:
        fig, axs = plt.subplots(2,3,figsize=(10,4))
        key=title.split('_')[0]
        try:
            test_input = test[key]['test'][int(title.split('.')[0][-1])]['input']
        except:
            test_input = test[key]['train'][9-int(title.split('.')[0][-1])]['input']
        task_data = np.array(task_data)
        prob_grid = np.array(prob_grid)
        prob_grid_exp = np.exp(prob_grid)
        # prob_grid_linrescale = (prob_grid - prob_grid.min()) / (prob_grid.max() - prob_grid.min())
        # prob_grid_rescale = (prob_grid_exp - prob_grid_exp.min()) / (prob_grid_exp.max() - prob_grid_exp.min())
        plt.suptitle(title, fontsize=8)

        task_data_shape = task_data.shape
        x,y = task_data_shape
        num_rot = title.count('rot90')
        if num_rot in [1,3] or 'transpose' in title:
            if num_rot in [1,3] and 'transpose' in title:
                direction_grid = np.arange(x*y).reshape(x,y)
            else:
                direction_grid = np.arange(x*y).reshape(y,x)
        else:
            direction_grid = np.arange(x*y).reshape(x,y)
        if 'transpose' in title:
            direction_grid = np.swapaxes(direction_grid, 0, 1)
        direction_grid = np.rot90(direction_grid,k=-num_rot)
        
    elif score is not None:
        if key:
            r = (len(score)+1)//4+1
            try:
                test_input = test[key.split('_')[0]]['test'][int(key.split('.')[0][-1])]['input']
            except:
                test_input = test[key.split('_')[0]]['train'][9-int(key.split('.')[0][-1])]['input']
        else:
            r = len(score)//4+1
        fig, axs = plt.subplots(r, 4, figsize=(10,2*r))
        
        
    else:
        # Extract examples
        train_examples = task_data.get('train', [])
        test_examples = task_data.get('test', [])
        original_test = task_data.get('original_test', [])
        trinf_examples = task_data.get('trinf_test', [])
    
        # Concatenate test examples in preferred order
        test_examples = trinf_examples + test_examples + original_test
    
        # Adjust task_solutions if TRINF examples are present
        if trinf_examples and task_solutions:
            task_solutions = [None] + task_solutions
    
        num_train = len(train_examples)
        num_test = len(test_examples)
        cols = num_train + num_test + (1 if answer else 0)
    
        # Resize if gridlines are requested
        if grid:
            figsize = (int(figsize[0] * 1.5), int(figsize[1] * 1.5))
    
        fig, axs = plt.subplots(2, cols, figsize=figsize)
        plt.suptitle(title, fontsize=12)
    fontsize = 8

    def draw_grid(ax, grid_data, title_str, prob_grid=None, inv=False, alpha=False):
        # ax.imshow(grid_data, cmap=ARC_COLORMAP, norm=ARC_NORM)
        height, width = len(grid_data), len(grid_data[0])
        ax.set_title(f"{title_str}({height},{width})", fontsize=fontsize)

        grid_data = np.array(grid_data)
        grid_data[grid_data==-1] = 10

        if alpha:
            # Draw base grid
            base_im = ax.imshow(grid_data, cmap=ARC_COLORMAP, norm=ARC_NORM)
        
            if prob_grid is not None:
                # Normalize alpha: high prob = low opacity
                if inv:
                    prob_grid = 1.0 - prob_grid  # inverse
                alpha_mask = np.clip(prob_grid, 0.0, 1.0)

                base = ax.imshow(np.ones(grid_data.shape,dtype=int)*10, cmap=ARC_COLORMAP, norm=ARC_NORM)
                overlay = ax.imshow(grid_data, cmap=ARC_COLORMAP, norm=ARC_NORM, alpha=alpha_mask)

        else:
            try:
                # Convert color-mapped indices to RGB image
                rgba = ARC_COLORMAP(ARC_NORM(grid_data))[:, :, :3]  # shape (H, W, 3)
            except:
                print(grid_data)
        
            # Apply brightening using prob_grid
            if prob_grid is not None:
                # print('rgba_before',rgba)
                prob_grid = np.clip(prob_grid, 0, 1)
                prob_grid *= 0.8
                # prob_grid = (prob_grid - prob_grid.min()) / (prob_grid.max() - prob_grid.min())
                if inv:
                    prob_grid = 1-prob_grid
                prob_grid = prob_grid[:, :, np.newaxis]  # shape (H, W, 1)
                # print('prob_grid newaxis',prob_grid)
                rgba = rgba * (1 - prob_grid) + prob_grid * 1.0  # brightening toward white
                # print('rgba_after',rgba)
        
            # Show the modified RGB image
            ax.imshow(rgba)
            
        if grid:
            ax.set_xticks(np.arange(width) - 0.5)
            ax.set_yticks(np.arange(height) - 0.5)
            ax.set_xticklabels([])
            ax.set_yticklabels([])
            ax.grid(True, color='white' if prob_grid is None else 'black', linewidth=0.2)
        else:
            ax.axis('off')

    if prob_grid is not None:
        draw_grid(axs[0,0], test_input, 'Test input')
        draw_grid(axs[1,0], answer, 'Target')
        
        def draw_prob_heatmap(ax, prob_grid, title_str="Probability Heatmap", fontsize=fontsize):
            im = ax.imshow(prob_grid, cmap='viridis')
            height, width = len(prob_grid), len(prob_grid[0])
            ax.set_title(title_str, fontsize=fontsize)
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            if grid:
                ax.set_xticks(np.arange(width) - 0.5)
                ax.set_yticks(np.arange(height) - 0.5)
                ax.set_xticklabels([])
                ax.set_yticklabels([])
                ax.grid(True, color='black', linewidth=0.2)
            else:
                ax.axis('off')
                
            # Add text annotations from direction_grid if provided
            if direction_grid is not None:
                # Normalize colormap to match imshow
                norm = im.norm
                cmap = im.cmap
        
                for i in range(height):
                    for j in range(width):
                        val = direction_grid[i][j]
                        if val in [0, 1]:
                            # Get background color from colormap
                            bg_color = cmap(norm(prob_grid[i][j]))
                            # Compute luminance (perceived brightness)
                            luminance = 0.299 * bg_color[0] + 0.587 * bg_color[1] + 0.114 * bg_color[2]
                            text_color = 'black' if luminance > 0.5 else 'white'
        
                            ax.text(j, i, str(val), ha='center', va='center', color=text_color, fontsize=fontsize - 2)

        draw_grid(axs[0,1], task_data, 'Predict')
        draw_prob_heatmap(axs[0,2],prob_grid,title_str='logsoftmax')
        draw_grid(axs[1,1], task_data, 'Predict Prob', prob_grid=prob_grid_exp)
        draw_prob_heatmap(axs[1,2],prob_grid_exp,title_str='prob')
        
    elif score is not None:
        if answer:
            score = ['Answer'] + score
            task_data = [answer] + task_data
        if key:
            score = ['Input'] + score
            task_data = [test_input] + task_data
        for i, ax in enumerate(axs.flat):
            try:
                draw_grid(ax, task_data[i], score[i])
            except:
                ax.axis('off')
    else:
        # Plot training inputs and outputs
        for i, example in enumerate(train_examples):
            draw_grid(axs[0, i], example['input'], "Train Input")
            draw_grid(axs[1, i], example['output'], "Train Output")
    
        # Plot test inputs and predicted or actual outputs
        for i, example in enumerate(test_examples):
            col_idx = num_train + i
            label = "TRINF Input" if (trinf_examples or original_test) and i == 0 else "Test Input"
            draw_grid(axs[0, col_idx], example['input'], label)
    
            try:
                if 'output' in example:
                    draw_grid(axs[1, col_idx], example['output'], "Test Output")
                elif task_solutions:
                    draw_grid(
                        axs[1, col_idx],
                        task_solutions[i],
                        "TRINF Output" if (trinf_examples or original_test) and i == 0 else "Test Output"
                    )
                else:
                    axs[1, col_idx].set_title("Test Output: ?", fontsize=fontsize)
                    axs[1, col_idx].axis('off')
            except Exception:
                axs[1, col_idx].set_title("Test Output: ?", fontsize=fontsize)
                axs[1, col_idx].axis('off')
    
        # Plot provided answer if any
        if answer:
            col_idx = num_train + num_test
            axs[0, col_idx].axis('off')
            draw_grid(axs[1, col_idx], answer[task_solutions.count(None)], "Answer")

    plt.tight_layout()
    if save:
        plt.savefig(f"{save}.png", bbox_inches="tight")
    plt.show()


In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
import hashlib
import os, sys
from tqdm import tqdm
from glob import glob
import itertools
import random
from visualize import *
from pp import *
from collections import deque
from copy import deepcopy
from collections import defaultdict

def cut_at_token(output, token_id):
    eos_positions = (output==token_id).nonzero()[0]
    return output[:eos_positions[0]] if len(eos_positions) else output

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation) == list(range(10))
    a = np.asarray(a)
    
    if a.ndim == 3:
        if not invert:
            permutation = np.argsort(permutation)
        a = a[..., permutation]
    
    else:
        assert a.ndim == 2
        if invert:
            permutation = np.argsort(permutation)
        perm_array = np.asarray(permutation)

        # Only apply permutation to elements not equal to -1
        mask = (a != -1) & (a != 10)
        a[mask] = perm_array[a[mask]]

    return a

def permute_rnd_col_(query):
    permutation = [0]+(1+np.random.permutation(9)).tolist()
    return 'permute' + ''.join(map(str, permutation))

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))

def permute_cnt_col_(query):
    elements, frequency = np.unique(np.concatenate([list(range(10))]+[np.array(x['input']).ravel() for x in query['train']]), return_counts=True)
    permutation = [0]+sorted(np.random.permutation(9)+1, key=lambda i: frequency[i], reverse=True)  # randomness as tie breaker
    return 'permute' + ''.join(map(str, permutation))

def permute_cnt_all_(query):
    elements, frequency = np.unique(np.concatenate([list(range(10))]+[np.array(x['input']).ravel() for x in query['train']]), return_counts=True)
    permutation = sorted(np.random.permutation(10), key=lambda i: frequency[i], reverse=True)  # randomness as tie breaker
    return 'permute' + ''.join(map(str, permutation))

permute_rnd_col = (permute_mod, permute_rnd_col_)
permute_rnd_all = (permute_mod, permute_rnd_all_)
permute_cnt_col = (permute_mod, permute_cnt_col_)
permute_cnt_all = (permute_mod, permute_cnt_all_)
permute_None = (np.copy, None)

class ArcDataset(object):
    # ----------------------------
    # OBJECT FINDING
    # ----------------------------
    def find_objects(self,grid,diagonal=True,bg=None):
        """Extract connected components and row-stack them"""
        grid = np.array(grid)
        h, w = grid.shape
        visited = np.zeros_like(grid, dtype=bool)

        if diagonal:
            directions = [(1,0), (-1,0), (0,1), (0,-1),
                          (1,1), (1,-1), (-1,1), (-1,-1)]
        else:
            directions = [(1,0), (-1,0), (0,1), (0,-1)]

        objects = []

        def bfs(sr, sc):
            q = deque([(sr, sc)])
            coords = []
            values = []
            v = grid[sr, sc]
            visited[sr, sc] = True
            while q:
                r, c = q.popleft()
                coords.append((r, c))
                values.append(grid[r, c])
                for dr, dc in directions:
                    nr, nc = r+dr, c+dc
                    if 0 <= nr < h and 0 <= nc < w and not visited[nr, nc] and grid[nr, nc] == v:
                        visited[nr, nc] = True
                        q.append((nr, nc))
            if bg is not None and v==bg: return
            # normalize
            min_r, min_c = min(rr for rr, _ in coords), min(cc for _, cc in coords)
            rel_coords = [(rr-min_r, cc-min_c) for rr, cc in coords]
            obj_h = max(rr for rr, _ in rel_coords) + 1
            obj_w = max(cc for _, cc in rel_coords) + 1
            obj_grid = np.full((obj_h, obj_w), -1, dtype=int)
            for (rr, cc), val in zip(rel_coords, values):
                obj_grid[rr, cc] = val
            return obj_grid

        for r in range(h):
            for c in range(w):
                if not visited[r, c]:
                    obj = bfs(r, c)
                    if bg is not None and grid[r][c]==bg:
                        continue
                    objects.append(obj)
        return objects

    # ----------------------------
    # PER-EXAMPLE BACKGROUND DETECTION
    # ----------------------------
    def detect_background(self, example_item):
        """
        Detect background value for a single example item (dict with 'input'/'output').
        Returns background value if full-size object exists, else None.
        """
        candidate = None
        for gkey in ["input", "output"]:
            if gkey in example_item:
                grid = np.array(example_item[gkey])
                objs = self.find_objects(grid)
                full_objs = [o for o in objs if o.shape == grid.shape]
                if len(full_objs)>1: return None
                if not full_objs:
                    if 'output' in example_item and gkey=='input' and grid.shape!=np.array(example_item['output']).shape: return None
                    continue  # no full-grid object
                val = np.unique(full_objs[0][full_objs[0] != -1])
                if len(val) == 1:
                    row_mask = np.where(np.all(full_objs[0] == val[0], axis=1))[0]
                    uniq_diff = np.unique(np.diff(row_mask))
                    if len(row_mask)>2 and len(uniq_diff)==1 and uniq_diff!=1: return None
                    col_mask = np.where(np.all(full_objs[0] == val[0], axis=0))[0]
                    uniq_diff = np.unique(np.diff(col_mask))
                    if len(col_mask)>2 and len(uniq_diff)==1 and uniq_diff!=1: return None
                    if candidate is None:
                        candidate = val[0]
                        if 'output' not in example_item: return candidate
                        elif gkey=='input' and grid.shape!=np.array(example_item['output']).shape: return candidate
                    elif candidate != val[0]:
                        return None  # inconsistent values
                else:
                    return None  # multiple values in full-grid object
        return candidate

    # ----------------------------
    # PAD OBJECTS VERTICALLY
    # ----------------------------
    def pad_between_objects(self, objects, pad_val):
        """Stack objects vertically with one row of padding between them."""
        if not objects:
            return np.array([[pad_val]])

        max_w = max(obj.shape[1] for obj in objects)
        padded = []

        for i, obj in enumerate(objects):
            # pad width to max_w
            h, w = obj.shape
            obj_padded = np.pad(obj, ((0, 0), (0, max_w - w)), constant_values=pad_val)
            obj_padded[obj_padded==-1] = pad_val
            padded.append(obj_padded)

            # add one row of padding between objects
            if i < len(objects) - 1:
                sep = np.full((1, max_w), pad_val)
                padded.append(sep)

        return np.vstack(padded)

    # ----------------------------
    # CUT BACKGROUND LINES/COLUMNS
    # ----------------------------
    def collapse_bg_lines(self, grid, bg_val=0):
        # --- rows ---
        row_is_bg = np.all(grid == bg_val, axis=1)
        keep_rows = []
        prev_bg = False
        for i, is_bg in enumerate(row_is_bg):
            if not is_bg:
                keep_rows.append(i)  # always keep non-bg row
                prev_bg = False
            else:
                if not prev_bg:
                    keep_rows.append(i)  # keep only first of a bg run
                prev_bg = True
        
        # grid = grid[keep_rows, :]
    
        # --- cols ---
        col_is_bg = np.all(grid == bg_val, axis=0)
        keep_cols = []
        prev_bg = False
        for j, is_bg in enumerate(col_is_bg):
            if not is_bg:
                keep_cols.append(j)
                prev_bg = False
            else:
                if not prev_bg:
                    keep_cols.append(j)
                prev_bg = True
    
        # grid = grid[:, keep_cols]
        return keep_rows, keep_cols

    def cut_background(self, in_grid, out_grid, bg_val):
        """Cut rows/cols consisting only of bg_val according to rules."""
        in_grid, out_grid = np.array(in_grid), np.array(out_grid)

        if in_grid.shape != out_grid.shape:
            keep_rows, keep_cols = self.collapse_bg_lines(in_grid,bg_val)
            in_cut = in_grid[keep_rows,:]
            in_cut = in_cut[:,keep_cols]
            out_cut = out_grid
        else:
            in_keep_rows, in_keep_cols = self.collapse_bg_lines(in_grid,bg_val)
            out_keep_rows, out_keep_cols = self.collapse_bg_lines(out_grid,bg_val)
            keep_rows = sorted(set(in_keep_rows)|set(out_keep_rows))
            keep_cols = sorted(set(in_keep_cols)|set(out_keep_cols))
            in_cut = in_grid[keep_rows,:]
            in_cut = in_cut[:,keep_cols]
            out_cut = out_grid[keep_rows,:]
            out_cut = out_cut[:,keep_cols]

        return in_cut, out_cut

    # ----------------------------
    # MAKE OBJECT DATASET
    # ----------------------------
    def make_object_dataset(self, make=False, pad='bg'):
        """
        Build object dataset: only objects stacked, one row of padding.
        Original grid excluded. Uses per-example background for padding.
        """
        obj_ds = defaultdict(lambda: {"train": [], "test": []})
        keys = []

        for key, example in self.queries.items():
            keys.append(key)
            for split in ["train", "test"]:
                for idx, item in enumerate(example.get(split, [])):
                    bg_val = self.detect_background(item)
                    pad_val = self.background_values[(key, split, idx)] = bg_val if bg_val is not None else -1
                    if pad==-1:pad_val=-1

                    if pp and make:
                        new_item = {}
                        for gkey in ["input", "output"]:
                            if gkey in item:
                                objs = self.find_objects(item[gkey],bg=bg_val)
                                full_grid_shape = np.array(item[gkey]).shape
                                # exclude object covering full original grid
                                # objs = [o for o in objs if o.shape != full_grid_shape]
    
                                if objs:
                                    obj_grid = self.pad_between_objects(objs, pad_val)
                                else:
                                    obj_grid = np.array([[pad_val]])
    
                                new_item[gkey] = obj_grid.tolist()
                        obj_ds[key][split].append(new_item)

        # return obj_ds
        if pp and make:
            return self.__class__(queries=obj_ds, keys=keys, background_values=self.background_values)
        else:
            return self.__class__(queries=self.queries, keys=keys, background_values=self.background_values)

    # ----------------------------
    # MAKE BACKGROUND-CUT DATASET
    # ----------------------------
    def make_background_cut_dataset(self):
        """
        Build background-cut dataset. Removes background-only rows/cols
        using per-example background values.
        """
        bg_cut_ds = defaultdict(lambda: {"train": [], "test": []})
        keys = []
        # print(self.background_values)
        
        for key, example in self.queries.items():
            keys.append(key)
            for split in ["train", "test"]:
                for idx, item in enumerate(example.get(split, [])):
                    bg_val = self.background_values.get((key, split, idx), None)
                    if bg_val is None:
                        # break
                        # no background detected, copy original
                        new_item = deepcopy(item)
                    else:
                        if split == "train":
                            new_in, new_out = self.cut_background(item["input"], item["output"], bg_val)
                            new_item = {"input": new_in.tolist(), "output": new_out.tolist()}
                        else:
                            new_item = deepcopy(item)
                    bg_cut_ds[key][split].append(new_item)

        return self.__class__(queries=dict(bg_cut_ds), keys=keys, background_values=self.background_values)

    @staticmethod
    def forward_mod(a, key, use_perm=True, is_output=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if op.startswith('I'):
                if is_output: continue
                op = op[1:]
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('dif'):     a[a==-1] = int(op[-1])
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('c'):       a = a
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('fix'):     a = a
            elif op.startswith('ice'):     a = a  # for adding icecuber solutions
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True, is_output=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if op.startswith('I'):
                if is_output: continue
                op = op[1:]
            if   op=='rot90':              a = np.rot90(np.rot90(np.rot90(a)))
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('dif'):     a[a==int(op[-1])] = -1
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('c'):       a = a
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('fix'):     a = a
            elif op.startswith('ice'):     a = a  # for adding icecuber solutions
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False, is_fake=False, background_values={}):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.is_fake = is_fake
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.faulty = {}
        self.transposed_dataset = None
        self.background_values = background_values

    @classmethod
    def empty(cls):
        return cls(queries={}, replies={}, keys=[])

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_fake=self.is_fake, is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, start=0, limit_n=20, get_diff=False):
        import numpy as np
        print(f"*** Load challanges from '{queries_file}'...")
        with open(queries_file) as f: queries = f.read()
        import os
        if os.getenv('KAGGLE_IS_COMPETITION_RERUN'): #Real submit
            is_fake = False
        else: #Fake run
            is_fake = True
        #is_fake = hashlib.md5(queries.encode('utf-8')).hexdigest().lower()=='a6b7dac3cab03abf2eb333e16610d6dc'
        if is_fake: print("*** -> Fake test set detected, setting flag 'is_fake' to True.")
        queries = json.loads(queries)
        if is_fake:
            start = start
            limit_n = limit_n
            print(f'From {len(queries)} to load only {limit_n}')
            # queries = {k:v for n,(k,v) in enumerate(queries.items()) if start <= n < start + limit_n}
            with open("/kaggle/input/arc2024ev-sol/arc-agi_evaluation_solutions.json",'r') as f:
                ev2024 = json.load(f)
            only2025 = {}
            for i,(k,v) in enumerate(queries.items()):
                # if i < start:
                #     continue
                if k not in ev2024:
                    only2025[k] = v
            print('only2025 len',len(only2025))
            new_queries = {}
            if 'training' not in queries_file: 
                queries = only2025
            only2025count = 0
            non_det_size_count = 0
            has_same_value_count = 0

            for i,(k,v) in enumerate(queries.items()):
                if i < start:
                    continue
                if len(new_queries)==limit_n:
                    break
                if get_same_value_count:
                    all_ok = True
                    shared_counts_per_train = []
                
                    for train in v['train']:
                        in_arr = np.array(train['input'])
                        out_arr = np.array(train['output'])
                
                        shared_counts = {}
                        for val in range(10):
                            in_count = np.sum(in_arr == val)
                            out_count = np.sum(out_arr == val)
                            if in_count > 0 and in_count == out_count:
                                shared_counts[val] = in_count
                
                        if not shared_counts:
                            all_ok = False
                            break
                        shared_counts_per_train.append(shared_counts)
                
                    if all_ok and k not in ev2024:
                        has_same_value_count += 1
                        new_queries[k] = v
                        # print(f"{k}: {shared_counts_per_train}")
                        # visualize_task(k, title=f'{k}:{shared_counts_per_train}')
                    continue
                # get_det_size = True
                # if get_det_size:
                    # if not all(np.array(w['input']).shape==np.array(w['output']).shape for w in v['train']):
                    #     non_det_size_count += 1
                    #     continue
                    # if not all(np.array(w['output']).shape==np.array(v['train'][0]['output']).shape for w in v['train']):
                    #     non_det_size_count += 1
                    #     continue
                if k not in ev2024 or 'training' in queries_file:
                    if get_diff and all(np.array(w['input']).shape==np.array(w['output']).shape for w in v['train']):
                        for t in v['train']:
                            in_arr = np.array(t['input'])
                            out_arr = np.array(t['output'])
                            out_arr[out_arr==in_arr] = -1
                            t['output'] = out_arr.tolist()
                        
                    new_queries[k] = v
                    if k not in ev2024:
                        only2025count += 1
            queries = new_queries
            if pp:print('loaded',len(queries),'of which only 2025 is',only2025count,'skipped non_det_size',non_det_size_count,'same_value_count',has_same_value_count)
        return cls(
            queries=queries,
            is_fake=is_fake,
            is_orig=True,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self, train=False):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        if not train:
            return self.__class__(
                keys=[f'{k}_{i}' for k, i in key_indices],
                queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
                replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
            )
        else:
            new_queries = {}
            new_replies = {}
            new_keys = []
        
            for k,i in key_indices:
                bk,aug = k.split('.',1)
                bk = bk.split('_')[0]
                t_ind = int(aug[-2])
                new_k = f'{bk}_{9-t_ind}.{aug}'
                new_queries[new_k] = {
                    'train': self.queries[k]['train'],
                    'test': [self.queries[k]['test'][i]],
                }
                new_replies[new_k] = [self.replies[k][i]]
                new_keys.append(new_k)
        
            return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def move_test_to_train(self, only_when_2=False):
        new_queries = {k: {'train': self.queries[k]['train'] if only_when_2 and len(self.queries[k]['train'])>2 else self.queries[k]['train'] + [{**t, 'output': self.replies[k][i]} for i, t in enumerate(self.queries[k]['test'])], 'test': []} for k in self.keys}
        return self.__class__(queries=new_queries, keys=[k for k in self.keys])

    def last_train_ex_for_test(self):
        assert not self.replies
        new_keys = [k+'t' if 'ex' in k else k for k in self.keys]
        new_queries = {k+'t' if 'ex' in k else k: {'train': self.queries[k]['train'][:-1], 'test': [{'input': self.queries[k]['train'][-1]['input']}]} for k in self.keys}
        new_replies = {k+'t' if 'ex' in k else k: [self.queries[k]['train'][-1]['output']] for k in self.keys}
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def length(self):
        return len(self.keys)

    def shuffled(self, seed=None):
        if seed is not None: np.random.seed(seed)
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def sorted(self, **kwargs):
        return self.__class__(queries=self.queries, replies=self.replies, keys=sorted(self.keys, **kwargs))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def sort_ex_by_input_size(self, seed=42, reverse=False):
        np.random.seed(seed)
        sort_key = lambda ex: np.prod(np.shape(ex['input']))
        new_queries = {k2: {k: (sorted(np.random.permutation(np.array(v, dtype=object)), key=sort_key, reverse=reverse) if k=='train' else v) for k, v in v2.items()} for k2, v2 in self.queries.items()}
        return self.__class__(queries=new_queries, replies=self.replies, keys=[k for k in self.keys])

    def interleave(self, block_size, num_gpus=None):
        keys = np.reshape(self.keys, (-1, block_size)).T
        if num_gpus is None: return self.change_keys(keys.ravel().tolist())
        ret, num_gpus = (None, num_gpus) if isinstance(num_gpus, int) else num_gpus
        keys = np.concatenate([keys, np.full((-keys.shape[0]%num_gpus, keys.shape[1]), None)])
        keys = np.reshape(keys, (keys.shape[0]//num_gpus, num_gpus, -1)).swapaxes(0, 1).reshape(num_gpus, -1)
        new_datasets = [self.change_keys(gpu_keys.tolist()) for gpu_keys in keys]
        return new_datasets if ret is None else new_datasets[ret]

    def remove(self, *datasets):
        remove_keys = {k for d in datasets for k in d.keys}
        new_keys = [k for k in self.keys if k not in remove_keys]
        return self.change_keys(new_keys)

    def keep_key_startswith(self, key_start):
        new_keys = [k for k in self.keys if k.startswith(key_start)]
        return self.change_keys(new_keys)

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only, esc=False):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            esc_id = np.random.randint(0,10) if esc and 'perm' not in desc else -1
            # if pp:print('k1',k1,'desc',desc, 'esc_id',esc_id)
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if (t=='input' or not inputs_only) and not(m=='train' and n==esc_id) else a) for t, a in x.items()} for n,x in enumerate(e)] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
            # if 0<=esc_id<len(queries[k1]['train']) and pp:
            #     import rich
        # if pp:print(queries[k1])
            #     visualize_task(queries[k1])
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False, esc=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only, esc=esc)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter):
        # if pp:print('get',self.queries[key]['train'])
        assert formatter.out2_token is None or key in self.replies
        train = formatter.fmt_train(self.queries[key]['train'], key=key.split('.c')[-1].replace('i','') if '.c' in key else None)
        query = formatter.fmt_query(self.queries[key]['test'], i=len(self.queries[key]['train']), key=key.split('.c')[-1].replace('i','') if '.c' in key else None)
        reply = formatter.fmt_reply(self.replies[key], self.faulty.get(key)) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True, key=key.split('.c')[-1].replace('i','') if '.c' in key else None)
        if random.randint(0,3000)==0 and pp:print('get text',key,text)
        # elif pp:print('get train valid',key)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter):
        return [self.get(key, formatter) for key in self.keys]

    def as_dataset(self):
        from datasets import Dataset
        return Dataset.from_list([{'key': k, 'query': self.queries[k], 'reply': self.replies[k]} for k in self.keys])

    def get_length(self, key, formatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer(ds.get(key, formatter=formatter)[name])['input_ids']) for ds in datasets)

    def get_lengths(self, formatter, name, max_of_transposed=False):
        return {key: self.get_length(key, formatter=formatter, name=name, max_of_transposed=max_of_transposed) for key in self.keys}

    def sorted_by_len(self, reverse=False, **kwargs):
        new_keys = [key for _, key in sorted([(v, k) for k, v in self.get_lengths(**kwargs).items()], reverse=reverse)]
        return self.change_keys(new_keys)

    def filter_by_len(self, min_len=0, max_len=float('inf'), **kwargs):
        new_keys = [k for k, v in self.get_lengths(**kwargs).items() if min_len<=v<=max_len]
        return self.change_keys(new_keys)

    import numpy as np
    import random
    
    def one_color(self, seed=0):
        rng = random.Random(seed)
        new_queries = {}
        new_replies = {}
        new_keys = []
        # new_queries.update(self.queries)
        # new_replies.update(self.replies)
        # new_keys.extend(self.keys)
    
        for n,key in enumerate(self.keys):
            if rng.choice([0,1]):
                new_queries[key] = self.queries[key]
                if key in self.replies:
                    new_replies[key] = self.replies.get(key,{})
                new_keys.append(key)
                continue
                
            train_colors = []
            filtered_trains = []
            new_key = key
                
            for example in self.queries[key]['train']:
                in_arr = np.array(example['input'])
                most_frequent = np.bincount(in_arr.flatten()).argmax()

                out_arr = np.array(example['output'])
                diff_arr = out_arr.copy()
                if in_arr.shape==out_arr.shape:
                    diff_arr[diff_arr==in_arr] = 0
                attention_colors = [v for v in list(np.unique(diff_arr)) if v not in [0,most_frequent,key.split('permute')[-1].find('0')]]
                # if pp:print('attention_colors',attention_colors)

                in_filter = False
                if attention_colors:
                    c = rng.choice(attention_colors)
                    train_colors.append(str(c))
                    in_new = in_arr.copy()
                    out_new = out_arr.copy()
                    if rng.choice([0]):
                        in_filter = True
                        train_colors.append('i')
                        in_new[in_new != c] = 0
                    out_new[out_new != c] = 0
                    filtered_trains.append({
                        'input': in_new.tolist(),
                        'output': out_new.tolist()
                    })
                else:
                    train_colors.append('_')
                    filtered_trains.append(None)  # Mark as skipped
    
            if all(t is None for t in filtered_trains):
                new_queries[key] = self.queries[key]
                if key in self.replies:
                    new_replies[key] = self.replies.get(key,{})
                new_keys.append(key)
                # if pp:print('No attention colors',key)
                # if pp:visualize_task(key)
                continue  # skip this key if no usable train
    
            # Clean color string: e.g., c1_4
            color_str = ''.join(train_colors)
            color_str = 'c' + ''.join(train_colors)
            new_key = f'{key}.{color_str}'
    
            # Process test
            test_list = self.queries[key]['test']
            reply_list = self.replies.get(key, [])

            filtered_replies = []

            for idx, test_example in enumerate(test_list):
                in_arr = np.array(test_example['input'])
                most_frequent = np.bincount(in_arr.flatten()).argmax()
    
                if idx < len(reply_list):
                    out_arr = np.array(reply_list[idx])
                    diff_arr = out_arr.copy()
                    if in_arr.shape==out_arr.shape:
                        diff_arr[diff_arr==in_arr] = 0
                    attention_colors = [v for v in list(np.unique(diff_arr)) if v not in [0,most_frequent,key.split('permute')[-1].find('0')]]
                    # if pp:print('test attention_colors',attention_colors)
                else:
                    attention_colors = []
    
                if not attention_colors:
                    continue  # skip test if no colors
                if rng.choice([0]):
                    continue
    
                c_test = rng.choice(attention_colors)
                test_color_str = f't{c_test}'
                if idx==0:
                    new_key += f'{test_color_str}'
                else:
                    new_key += f'{c_test}'
    
                # Add reply if exists
                if idx < len(reply_list):
                    out_arr = np.array(reply_list[idx])
                    out_new = out_arr.copy()
                    out_new[out_new != c_test] = 0
                    filtered_replies.append(out_new.tolist())

            # if new_key==key:
            #     continue
                
            if new_key not in new_queries:
                new_queries[new_key] = {'train': [], 'test': test_list}
                new_keys.append(new_key)
                if reply_list and filtered_replies:new_replies[new_key] = []
    
            # Add filtered train examples
            for filtered in filtered_trains:
                if filtered is not None:
                    new_queries[new_key]['train'].append(filtered)

            for filtered in filtered_replies:
                if filtered is not None:
                    new_replies[new_key].append(filtered)
            # if pp:visualize_task(new_queries[new_key],new_replies[new_key],title=f'#{n},{new_key}')

        # if pp:print('new_keys',new_keys)
        return self.__class__(
                keys=new_keys,
                queries=new_queries,
                replies=new_replies
            )


    def get_diff(self):
        import copy
        new_queries = {}
        new_replies = {}
        new_keys = []
        for k in self.keys:
            modified_key = k
            if all(np.array(v['input']).shape == np.array(v['output']).shape for v in self.queries[k]['train']):
                unique_values = set()
                for v in self.queries[k]['train']:
                    unique_values.update(np.unique(v['input']))
                    unique_values.update(np.unique(v['output']))
                for v in self.queries[k]['test']:
                    unique_values.update(np.unique(v['input']))
                remain_values = sorted(set(range(10)) - unique_values)
                if remain_values:
                    diff_marker = remain_values[-1]
                    modified_key = f'{k}.dif{diff_marker}'
                    
                    queries_copy = copy.deepcopy(self.queries[k])
                    replies_copy = copy.deepcopy(self.replies.get(k,{}))
    
                    for v in queries_copy['train']:
                        in_arr = np.array(v['input'])
                        out_arr = np.array(v['output'])
                        out_arr[out_arr == in_arr] = diff_marker
                        v['output'] = out_arr.tolist()
    
                    for i, v in enumerate(queries_copy['test']):
                        if len(replies_copy) > i:
                            in_arr = np.array(v['input'])
                            out_arr = np.array(replies_copy[i])
                            out_arr[out_arr == in_arr] = diff_marker
                            replies_copy[i] = out_arr.tolist()
                else:
                    queries_copy = self.queries[k]
                    replies_copy = self.replies.get(k,{})
            else:
                queries_copy = self.queries[k]
                replies_copy = self.replies.get(k,{})
    
            new_queries[modified_key] = queries_copy
            if replies_copy:
                new_replies[modified_key] = replies_copy
            new_keys.append(modified_key)
            
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
                    
    def cut_to_query_count(self, max_count, from_end=False, p=1):
        import copy
        import random
    
        new_queries = {}
        new_keys = []
        new_replies = copy.deepcopy(self.replies)
    
        for k in self.keys:
            if random.random() < p:
                # Rename key if needed
                if not from_end:
                    new_key = '.'.join([
                        v[:max_count + 2] if v.startswith('ex') else v
                        for v in k.split('.')
                    ])
                else:
                    new_key = '.'.join([
                        'ex' + v[-max_count:] if v.startswith('ex') else v
                        for v in k.split('.')
                    ])
                new_keys.append(new_key)
    
                # Copy replies if present
                if k in self.replies:
                    new_replies[new_key] = copy.deepcopy(self.replies[k])
    
                # Deep copy of queries[k]
                q = copy.deepcopy(self.queries[k])
                if 'train' in q and isinstance(q['train'], list):
                    if from_end:
                        q['train'] = q['train'][-max_count:]
                    else:
                        q['train'] = q['train'][:max_count]
                new_queries[new_key] = q
            else:
                new_keys.append(k)
                new_queries[k] = self.queries[k]
    
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def cut_to_len(self, formatter, name, max_len, max_new_tokens='auto', from_end=False, quiet=False, **kwargs):
        if max_new_tokens:
            if max_new_tokens=='auto': max_new_tokens = formatter.max_new_tokens()
            max_len_old, max_len = max_len, max_len - max_new_tokens
            if not quiet: print(f'*** Reducing task size to max. {max_len_old} tokens ({max_len} input + {max_new_tokens} generated)...')
        elif not quiet: print(f'*** Reducing task size to max. {max_len} tokens...')
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in (self.keys if quiet else tqdm(self.keys, file=sys.stdout)):
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name, **kwargs):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'): key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None: temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply

        # if pp:print('new_keys in cut_to_len',new_keys)
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)

    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            local_keep_max = keep_max
            if local_keep_max == 'rand':
                local_keep_max = np.random.randint(0, n + 1)  # 0 to n inclusive
            if local_keep_max is not None:
                p = p[:local_keep_max]
            new_key = f'{key}.ex' + ('-' if len(p) and (p.max() > 9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {
                k: (np.array(v, dtype=object)[p].tolist() if k == 'train' else v)
                for k, v in self.queries[key].items()
            }
            if key in self.replies:
                new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def shuffle_rp(self, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['test'])
            p = np.random.permutation(n)
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.rp' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='test' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = np.array(self.replies[key], dtype=object)[p].tolist()
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def append_to_keys(self, text):
        return self.change_keys([f'{k}{text}' for k in self.keys])

    def random_select(self, n):
        keys = np.array(self.keys).reshape(n, -1).T
        choice = np.random.randint(0, n, size=[len(keys)])
        return self.change_keys(keys[np.arange(len(keys)), choice])

    def augment(self, tp=False, rot=False, n=1, perm=None, perm_append=False, shfl_keys=False, shfl_ex=False, seed=None, quiet=False, inputs_only=False, esc=False, keep_max=None):
        if not quiet: print(f"*** Augment dataset{' (inputs only)' if inputs_only else ''}...")
        np.random.seed(seed)
        d = self
        if tp: d = d.mod(np.transpose, keep=True, inputs_only=inputs_only, esc=esc)
        if tp=='rand': d = d.random_select(n=2)
        if rot: d = d.mod(np.rot90, n=3, keep=True, inputs_only=inputs_only, esc=esc)
        if rot=='rand': d = d.random_select(n=4)
        if perm is None and n<=1: d = d.shuffled() if shfl_keys else d
        else: d = d.mod(*([np.copy] if perm is None else globals()[f"permute_{perm}"]), n=n, shuffle=shfl_keys, keep=perm_append, inputs_only=inputs_only)
        np.random.seed(seed)
        if shfl_ex: d = d.shuffle_ex(keep_max=keep_max)
        return d


    def one_rotp(self, p=1, n=4):
        import copy  # for deep copying replies safely

        new_queries = {}
        new_keys = []
        new_replies = copy.deepcopy(self.replies)  # avoid in-place mutation
    
        for key in self.keys:
            if random.random() < p:
                new_key = key + f'_rotpn{n}'
                new_queries[new_key] = {}
                if key in self.replies:new_replies[new_key] = copy.deepcopy(self.replies[key])
                for k, v in self.queries[key].items():
                    new_examples = []
                    for example in v:
                        input_arr = np.array(example['input'])
                        all_rows = []
                        for i in range(n):
                            rotated = np.rot90(input_arr, k=i % 4)  # ensure 0–3
                            all_rows.extend(rotated.tolist()+[[]])  # row-wise flatten
                        new_example = {'input': all_rows}
                        if 'output' in example:
                            new_example['output'] = example['output']
                        new_examples.append(new_example)
                    new_queries[new_key][k] = new_examples
                new_keys.append(new_key)
            else:
                new_queries[key] = self.queries[key]
                new_keys.append(key)
    
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def remove_replies(self):
        return self.__class__(queries=self.queries, replies={}, keys=[k for k in self.keys])

    def split_at_pos(self, pos=None, random_seed=None):
        keys = self.keys
        keys_split = [keys[i::4] for i in range(4)]

        return tuple(self.change_keys(new_keys, keep_flags=True) for new_keys in keys_split)

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            try:
                target_dict = submission[base_id][int(base_nr)]
                for i, g in enumerate(v[:len(target_dict)]):
                    
                    target_dict[f'attempt_{i+1}'] = g.tolist()
            except:
                pass

    def validate_submission(self, submission, queries_file):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            print(k)
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        print(score)
                        # print(r)
                        visualize_task(k,[None]*i + [r], file=queries_file)
                        # break
                    else:
                        if submission[k][i][attempt]==[[0]]:# and (attempt=='attempt_2' or i>0):
                            continue
                        print(attempt)
                        visualize_task(k,[None]*i + [submission[k][i][attempt]], file=queries_file)
                            # print(r)
                            # print(submission[k][i][attempt])
        return score
def get_class_MyDataCollator(cache=[]):
    if not cache:
        from trl import DataCollatorForCompletionOnlyLM
        class MyDataCollator(DataCollatorForCompletionOnlyLM):
            def setup(self, out2_token_id=None, fault_token_id=None, fault_freq=0, sample_tries=8, mask_first_output=False):
                self.out2_token_id = out2_token_id
                self.fault_token_id = fault_token_id
                self.fault_freq = fault_freq
                self.sample_tries = sample_tries
                self.mask_first_output = mask_first_output
                self.need_print = True
                return self

            def torch_call(self, examples):
                batch = super().torch_call(examples)
                if self.out2_token_id is not None:
                    # if pp:print('out2_token_id: batch:',batch)
                    assert not self.fault_freq
                    for i in range(len(batch['input_ids'])):
                        end_pos = ((batch['labels'][i] != -100              ).nonzero().max()).item() + 1
                        mid_pos = ((batch['labels'][i] == self.out2_token_id).nonzero().max()).item() + 1
                        beg_pos = mid_pos - (end_pos - mid_pos)
                        batch['labels'][i][beg_pos:mid_pos] = batch['labels'][i][mid_pos:end_pos]
                elif self.fault_freq:
                    # if pp:print('fault_freq',batch)
                    for i in range(len(batch['input_ids'])):
                        end_pos = ((batch['labels'][i] != -100).nonzero().max()).item() + 1
                        if not isinstance(self.fault_freq, float):
                            eos_token_id = batch['labels'][i][end_pos - 1]
                            num_examples = (batch['labels'][i] == eos_token_id).sum().item() - 1
                            fault_freq = self.fault_freq[num_examples]
                        else: fault_freq = self.fault_freq
                        if random.random() < fault_freq:
                            beg_pos = ((batch['labels'][i][:end_pos]==-100).nonzero().max()).item() + 1
                            fault_pos = random.randint(beg_pos, end_pos-2)
                            fault_tok = batch['labels'][i][fault_pos].item()
                            for t in range(self.sample_tries):
                                new_tok = batch['labels'][i][random.randint(beg_pos, end_pos-2)].item()
                                if fault_tok!=new_tok:
                                    batch['input_ids'][i][fault_pos] = new_tok
                                    batch['labels'][i][fault_pos+1:end_pos] = self.fault_token_id
                                    break
                # else:
                #     if pp:print('batch as is :',batch)
                for i in range(len(batch['labels'])):
                    for _ in range(self.mask_first_output):
                        beg_pos = ((batch['labels'][i] != -100).nonzero().min()).item()
                        mid_pos = ((batch['labels'][i][beg_pos:] == -100).nonzero().min()).item() + beg_pos
                        end_pos = ((batch['labels'][i] != -100).nonzero().max()).item() + 1
                        if mid_pos<end_pos: batch['labels'][i][beg_pos:mid_pos] = -100

                return batch
        cache.append(MyDataCollator)
    return cache[0]

class ArcFormatter(object):
    def __init__(self, inp_prefix, out_prefix, arr_sep, out2_use=False, out2_token=None, arr_beg='', arr_end='', pretext='', pre_out=None, exa_sep='', exa_end='', qry_prefix=None, rpl_prefix=None, rpl_sep=None, dec_sep=None, min_wid=0, min_pad='', pretext_corpus_split='', masking=0, tokenizer=None, collator_kwargs={}, repeat_input_aug=None, repeat_input_pre=None):
        self.tokenizer = tokenizer
        self.inp_prefix = inp_prefix
        self.out_prefix = out_prefix
        self.out2_token = out2_token
        self.out2_use = out2_use
        assert not out2_use or out2_token is not None
        assert not out2_use or masking in [1, 2]
        assert masking!=2 or out2_use or rpl_prefix is not None
        self.qry_prefix = qry_prefix if qry_prefix is not None else inp_prefix
        self.rpl_prefix = rpl_prefix if rpl_prefix is not None else out_prefix
        self.rpl_sep = rpl_sep if rpl_sep is not None else self.rpl_prefix
        self.arr_sep = arr_sep
        self.arr_beg = arr_beg
        self.arr_end = arr_end
        self.pretext = pretext
        self.pre_out = pre_out
        self.pre_out_empty = ['']*99
        self.pretext_corpus_split = pretext_corpus_split
        self.exa_sep = exa_sep
        self.exa_end = exa_end
        self.dec_sep = arr_sep if dec_sep is None else dec_sep
        self.min_wid = min_wid
        self.min_pad = min_pad
        self.masking = masking
        self.collator_kwargs = collator_kwargs
        self.repeat_input_aug = repeat_input_aug
        self.repeat_input_pre = repeat_input_pre

    def fmt_array(self, array):
        return self.arr_beg + self.arr_sep.join(str(row).replace(' ', '').replace(',', '').replace('[', '').replace(']', '')+self.min_pad*max(0, self.min_wid-len(row)) for row in array) + self.arr_end

    def get_pre_out(self, pretext_split):
        if self.pre_out is None: return self.pre_out_empty
        if pretext_split: return [self.pretext_corpus_split.join(list(p) + ['']) for p in self.pre_out]
        return self.pre_out

    def fmt_train(self, train, last_is_challenge=False, pretext_split=False, key=None):
        po = self.get_pre_out(pretext_split=pretext_split)
        ex = [(f"{self.fmt_query([x], i, pretext_split=pretext_split, key=key)}{self.fmt_reply([x['output']])}" if last_is_challenge and i+1==len(train) else
               f"{self.inp_prefix}{self.fmt_array(x['input'])}{self.repeat_input(x, no_aug=pretext_split)}{po[i]}{key[i] if key and key[i]!='_' else ''}{self.out_prefix}{self.fmt_array(x['output'])}") for i, x in enumerate(train)]
        pre = self.pretext_corpus_split.join(list(self.pretext)+['']) if pretext_split else self.pretext
        end = '' if last_is_challenge else (self.exa_end + self.tokenizer.eos_token)
        return pre + (self.exa_end + self.tokenizer.eos_token + self.exa_sep).join(ex) + end

    def fmt_query(self, query, i, pretext_split=False, key=None):
        po = self.get_pre_out(pretext_split=pretext_split)
        return ''.join(f"{self.qry_prefix}{self.fmt_array(x['input'])}{self.repeat_input(x, no_aug=pretext_split)}{po[i]}{key.split('.')[0][-1] if key and 't_.' not in key else ''}{self.rpl_prefix}" for x in query[:1])

    # def fmt_query(self, query, i, pretext_split=False, key=None):
    #     po = self.get_pre_out(pretext_split=pretext_split)
    #     return ''.join(f"{self.qry_prefix}{self.inp_prefix.join([self.fmt_array(x['input'][b*len(x['input'])//4:b+len(x['input'])//4]) for b in range(4)])}{self.repeat_input(x, no_aug=pretext_split)}{po[i]}{key.split('.')[0][-1] if key and 't_.' not in key else ''}{self.rpl_prefix}" for x in query[:1])

    def repeat_input(self, x, no_aug=False):
        if self.repeat_input_aug is None: return ''
        return f"{self.repeat_input_pre}{self.fmt_array(((lambda x: x) if no_aug else self.repeat_input_aug)(x['input']))}"

    def fmt_reply(self, reply, fault=None):
        ids = self.fmt_array(reply[0]) + self.exa_end + self.tokenizer.eos_token
        if self.out2_use:
            if fault is None: fault = reply
            ids = self.fmt_array(fault[0]) + self.exa_end + self.out2_token + ids

        return ids

    def quick_test(self, decoded, done):
        sp = decoded.split(self.tokenizer.eos_token)[0].split(self.dec_sep)
        sl = len(sp[0])
        is_prefix = sl>0 and len(sp[-1])<=sl and (len(sp)==1 or len(sp[-2])==sl) and all(x.isdigit() for x in sp[-1])
        return is_prefix and (not done or len(sp[-1])==0 or len(sp[-1])==sl)

    @staticmethod
    def is_valid_solution(guess):
        return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

    def max_new_tokens(self, safety_margin=1):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        max_sized_reply[:,::2] = 1
        tokenized = self.tokenizer(self.fmt_reply([max_sized_reply]))['input_ids']
        max_new_tokens = len(tokenized)
        if tokenized[0]==self.tokenizer.bos_token_id: max_new_tokens -= 1
        return max_new_tokens + safety_margin

    def de_tokenize(self, tokens, scores=None, classifier=None):
        import torch, copy
        if classifier is not None:
            de_tokenized = copy.deepcopy(tokens)
            tokens_cut = tokens = np.array(tokens).ravel()
            # if pp:print('tokens_cut',tokens_cut,'de_tokenized',de_tokenized, 'scores',scores)
        else:
            tokens_cut = cut_at_token(tokens, self.tokenizer.eos_token_id)
            de_tokenized = self.tokenizer.batch_decode([tokens_cut])[0]
        # if pp:print('detokenized',de_tokenized)
        score_val = None
        if scores is not None and len(tokens_cut)>0:
            tokens_with_eos = tokens[:len(tokens_cut)+1]
            score_val = torch.nn.functional.log_softmax(torch.tensor(scores), dim=-1).numpy().copy()[np.arange(len(tokens_with_eos)), tokens_with_eos].sum()
            # if pp:print(f'score_val{score_val:.1f} for len',len(de_tokenized),'detokenized ',de_tokenized,'\n=sum of len',len(tokens_with_eos),torch.nn.functional.log_softmax(torch.tensor(scores), dim=-1).numpy().copy().round(1)[np.arange(len(tokens_with_eos)), tokens_with_eos])
            # if pp:print(f'In detokenized, score_val{score_val:.1f} for len',len(de_tokenized),'detokenized ',de_tokenized)
            if classifier is not None:
                number_token_ids = list(range(11))
            else:
                number_token_ids = [self.tokenizer.vocab[k] for k in map(str, range(10))]
            fault_token_id = self.collator_kwargs.get('fault_token_id')
            if fault_token_id is not None: number_token_ids.append(fault_token_id)
            number_token_ids = np.array(number_token_ids)
            number_positions = (tokens_cut[..., np.newaxis] == number_token_ids).any(-1)
            scores = scores[:len(tokens_cut), number_token_ids][number_positions]
            scores = torch.nn.functional.log_softmax(torch.tensor(scores), dim=-1)[:, :10 if classifier is None else 11].numpy().copy()
        return max(len(tokens)+1, len(tokens_cut)), score_val, de_tokenized, scores

    def decode_to_array_single(self, text, score=None, limit_rows=30):
        # if pp:print('text',text)
        try:
            import traceback
            if isinstance(text,list):
                by_rows = text
                limited = False
            else:
                by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in text.split(self.dec_sep)] if len(row)]
                if limit_rows and len(by_rows) > limit_rows:
                    by_rows = by_rows[:limit_rows]
                    limited = True
                else: limited = False
            decoded = np.array(by_rows, dtype=int)
            if self.is_valid_solution(decoded):
                try:
                    assert score is not None
                    decoded_flat = decoded.ravel()
                    if limited: score = score[:len(decoded_flat)]
                    score_all = score.reshape(decoded.shape + score.shape[1:])
                    # if pp:print('decoded',decoded.shape,'decoded_flat',decoded_flat.shape,'score',score.shape,'score_all',score_all.shape)
                    score_result = score[range(len(decoded_flat)), decoded_flat]
                    score_reshaped = score_result.reshape(decoded.shape)
                    score_cum_reshaped = score_result.cumsum().reshape(score_reshaped.shape)
                    score_all_cum = score_cum_reshaped[..., np.newaxis] - score_reshaped[..., np.newaxis] + score_all
                except: 
                    if pp:print(traceback.format_exc())
                    score_reshaped = score_cum_reshaped = np.full(decoded.shape, -float('inf'))
                # if pp:print('output',decoded,'score',score_reshaped.shape,'score_cum',score_cum_reshaped.shape,'score_all',score_all.shape,'score_all_cum',score_all_cum.shape)
                return {'output': decoded, 'score': score_reshaped, 'score_cum': score_cum_reshaped, 'score_all': score_all, 'score_all_cum': score_all_cum}
        except: 
            pass
            # if pp:print('D not is_valid_solution',text,traceback.format_exc())
        if pp:print('D not is_valid_solution',text,traceback.format_exc())
        return {}

    def decode_to_array(self, text, score=None, limit_rows=30):
        if not self.out2_use: text, score = [text], [score]
        else:
            text = text.split(self.out2_token)
            if score is None: score = [None]*len(text)
            else:
                lengths = np.cumsum([len(list(filter(str.isdigit, t))) for t in text])
                score = [score[s:e] for s, e in zip([0]+lengths[:-1].tolist(), lengths)]
        return [self.decode_to_array_single(t, s) for t, s in zip(text, score)]

    def get_corpus(self):
        try:
            old_min_wid, self.min_wid = self.min_wid, min(self.min_wid, 2)
            return self.fmt_train([{'input': [[i] for i in range(10)], 'output': [[i] for i in range(10)]}]*3, last_is_challenge=True, pretext_split=True)
        finally: self.min_wid = old_min_wid

    def get_data_collator(self):
        if not self.masking: return None
        from transformers import DataCollatorForLanguageModeling
        collator_params = dict(tokenizer=self.tokenizer, mlm=False)
        pass_out2_token = self.tokenizer.vocab[self.out2_token] if self.out2_use and self.masking==1 else None
        # if pp:print('pass_out2_token',pass_out2_token)
        # if pp:print('vacab',self.tokenizer.vocab)
        if self.masking:
            assert not self.collator_kwargs.get('mask_first_output') or self.masking==1
            data_collator = get_class_MyDataCollator()(
                **collator_params,
                instruction_template=[self.inp_prefix, self.tokenizer.bos_token][self.masking - 1],
                response_template=[self.out_prefix, (self.out2_token if self.out2_use else self.rpl_sep)][self.masking - 1],
            ).setup(out2_token_id=pass_out2_token, **self.collator_kwargs)
        else:
            assert not self.collator_kwargs, 'only supported with masking on'
            data_collator = DataCollatorForLanguageModeling(**collator_params)
        return data_collator

    def get_output_token_ids(self):
        assert not self.out2_use
        num_tokens = [self.tokenizer.vocab[str(i)] for i in range(10)]
        sep_tokens = [tok for txt in [self.arr_beg, self.arr_sep, self.arr_end, self.exa_sep] if txt for tok in self.tokenizer(txt)['input_ids'][1:]]
        sep_tokens.append(self.tokenizer.eos_token_id)
        return num_tokens + sorted(set(sep_tokens))

ArcFormatter_pretext2 = lambda **kwargs: ArcFormatter(masking=1, inp_prefix='I', out_prefix='O', arr_sep='\n', arr_end='\n', pretext='ABCDEFGHJKLMNPQRSTUVWXYZ', pretext_corpus_split='\n', **kwargs)
ArcFormatter_pretext3 = lambda **kwargs: ArcFormatter(masking=1, inp_prefix='I', out_prefix='O', arr_sep='\n', arr_end='\n', pretext='ABCDEFGHJKLMNPQRSTUVWXYZabcdefghjklmnpqrstuvwxyz', pretext_corpus_split='\n', **kwargs)
ArcFormatter_premix_2 = lambda **kwargs: ArcFormatter(masking=1, inp_prefix='I', out_prefix='O', arr_sep='\n', arr_end='\n', pretext='ABCDEFGHJKLMNPQRSTUVWXYZ', pre_out=['+/-=']*99, pretext_corpus_split='\n', **kwargs)
ArcFormatter_premix_3 = lambda **kwargs: ArcFormatter(masking=1, inp_prefix='I', out_prefix='O', arr_sep='\n', arr_end='\n', pretext='ABCDEFGHJKLMNPQRSTUVWXYZabcdefghjklmnpqrstuvwxyz', pre_out=['+/-=']*99, pretext_corpus_split='\n', **kwargs)

available_formatters = dict(
    ArcFormatter_pretext2=ArcFormatter_pretext2,
    ArcFormatter_pretext3=ArcFormatter_pretext3,
    ArcFormatter_premix_2=ArcFormatter_premix_2,
    ArcFormatter_premix_3=ArcFormatter_premix_3,
)

In [ ]:
%%writefile selection.py
import numpy as np
from pp import *
from visualize import *

def hashable(guess):
    return tuple(map(tuple, guess))

def make_unique(guess_list, indices=None):
    used = set()
    out = []
    out_ind = []
    for i, g in enumerate(guess_list):
        h = hashable(g)
        if h not in used:
            used.add(h)
            out.append(np.array(g))
            if indices is not None: out_ind.append(indices[i])
    return out if indices is None else (out, out_ind)

def first_only(guesses):
    return [g['output'] for g in guesses.values()][:1]

def keep_order(guesses):
    return [g['output'] for g in guesses.values()]

def keep_order_unique(guesses):
    return make_unique(keep_order(guesses))

def get_best_shape_by_score(guess_list, getter, once_per_result=True):
    seen_outputs = set()
    shape_scores = {}
    for i, g in enumerate(guess_list):
        shape = tuple(g['output'].shape)
        scores = shape_scores[shape] = shape_scores.get(shape, [[], []])
        scores[1].append(i)
        h = hashable(g['output'])
        if h in seen_outputs: continue
        if once_per_result: seen_outputs.add(h)
        scores[0].append(g)
    shape_scores = [(getter(scores), shape, indices) for shape, (scores, indices) in shape_scores.items()]
    shape_scores = sorted(shape_scores, key=(lambda x: x[0]), reverse=True)
    if pp:print('shape_scores',shape_scores)
    return shape_scores[0]

def score_sum(guesses, getter, shape_getter=None, prefer_common_shape=True, ppp=False):
    if pp and ppp:print('\nguesses in score_sum, len:',len(guesses), guesses.keys())
    if shape_getter is None: shape_getter = getter
    guess_list = list(guesses.values())
    common_shape_indices = set(get_best_shape_by_score(guess_list, shape_getter)[2]) if prefer_common_shape else []
    scores = {}
    for i, g in enumerate(guess_list):
        h = hashable(g['output'])
        x = scores[h] = scores.get(h, [i in common_shape_indices, [], g['output']])
        x[1].append(g)
        if pp and ppp:print(sum(np.exp(v['score_val']) for v in x[1]),[v['score_val'] for v in x[1]],[np.exp(v['score_val']) for v in x[1]])
    scores = [(cs, getter(sc), o) for cs, sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[:2]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    if pp and ppp and guesses:
        ordered_scores = [(x[0],round(x[1],3)) for x in scores]
        print('ordered_scores in score_sum',ordered_scores)
        visualize_task(ordered_outputs,score=ordered_scores,key=list(guesses.keys())[0].split('.')[0])
    return ordered_outputs

getter_all_probsum = lambda guesses: sum(np.exp(g['score_val']) for g in guesses)
def score_all_probsum(guesses): return score_sum(guesses, getter_all_probsum, ppp=True)

def getter_full_probmul(p):
    def _getter(guesses, baseline=p):
        inf_score = sum([g['score_val']+baseline for g in guesses])
        # if pp:print('inf_score is sum of [g[score_val]+baseline for g in guesses]',[g['score_val']+baseline for g in guesses])
        try:
            aug_score = np.mean([sum(s+baseline for s in g['score_multi_nl']) for g in guesses])
            # if pp:print('aug_score is mean of [sum(s+baseline for s in g[score_multi_nl]) for g in guesses]',[sum(s+baseline for s in g['score_multi_nl']) for g in guesses])
            # if pp:print('inf_score',inf_score,'aug_score',aug_score)
        except:
            if pp:print('aug_score fail')
            aug_score = -1000
        return inf_score + aug_score
    return _getter

def score_full_probmul_3(guesses): return score_sum(guesses, getter_full_probmul(3), prefer_common_shape=False, ppp=use_aug)

selection_algorithms = [
    # first_only,
    # keep_order,
    # keep_order_unique,
    # score_all_probsum,
    score_full_probmul_3,
]

In [ ]:
%%writefile async_tools.py
import sys
import asyncio

async def stream_reader(stream, id, to):
    id = '' if id is None else f'{id}. '
    data = b''
    while True:
        read = await stream.read(n=4096)
        if not read: break
        if to is not None:
            *complete_lines, data = (data + read + b'X').splitlines()
            data = data[:-1]
            for line in complete_lines:
                line = line.rstrip()
                if line: print(f"{id}{line.decode('utf-8')}", file=to, end='\n', flush=True)

async def wait_for_subprocess(subprocess, print_output=False, id=None):
    await asyncio.gather(
            stream_reader(subprocess.stdout, id, (sys.stdout if print_output else None)),
            stream_reader(subprocess.stderr, id, (sys.stderr if print_output else None)),
        )
    return await subprocess.wait()

async def wait_for_subprocesses(*processes, print_output=False):
    return await asyncio.gather(*[wait_for_subprocess(p, print_output=print_output, id=i if len(processes)>1 else None) for i, p in enumerate(processes)])

In [ ]:
%%writefile pp.py
import os
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    pp = False
else:
    pp = True
cut_query = 2
rotp = False
det_size = True
c_head = False
if not c_head:det_size = False
fp32 = True
use_aug = True
use_cb = False
use_ext = False
get_same_value_count = False

In [ ]:
%%writefile common_stuff.py
# common configuration for training and evaluation
from pp import *
from arc_loader import *
from model_runner import *
from selection import *
from async_tools import *
import time
import os
import torch
import random
import numpy as np

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if using multi-GPU

# For reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

import traceback
gpu_count = torch.cuda.device_count()
print('gpu_count',gpu_count)
import warnings
warnings.filterwarnings("ignore")

# paths
tmp_dir = '/kaggle/temp'
# tmp_dir = '/kaggle/working'
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    arc_challenge_file = '/kaggle/input/arc-prize-2025/arc-agi_test_challenges.json'
else:
    arc_challenge_file = '/kaggle/input/arc-prize-2025/arc-agi_evaluation_challenges.json'
arc_solutions_file = '/kaggle/input/arc-prize-2025/arc-agi_evaluation_solutions.json'
model_temp_storage = os.path.join('', 'finetuned_model')
infer_temp_storage = os.path.join(tmp_dir, 'inference_outputs')
score_temp_storage = os.path.join(tmp_dir, 'inference_scoring')

keep_model = False
do_train = False
get_diff = False
one_rotp = False
one_ex = False
one_ex_mix = False
keep_max = 'rand' if one_ex_mix else None
train_classifier = False
if train_classifier:
    do_train = True
    arc_challenge_file = arc_challenge_file.replace('evaluation','training')
    arc_solutions_file = arc_solutions_file.replace('evaluation','training')
use_classifier = c_head

# load datasets
arc_test_set = ArcDataset.from_file(arc_challenge_file, start=300 if do_train else 0, limit_n=300 if do_train else 120)
obj_start_time = time.time()
if arc_test_set.is_fake: arc_test_set.load_replies(arc_solutions_file)
keys = arc_test_set.sorted_by_len(formatter=None, name='input').keys
new_keys = []
check_index = []
for n,k in enumerate(keys):
    if n%8<5:
        new_keys.append(k)
        check_index.append(n)
    else:
        new_keys.insert(4-n%8,k)
        check_index.insert(4-n%8,n)
# if pp:print('check_index',check_index)
keys = new_keys

if arc_test_set.is_fake and not train_classifier:
    keys = keys[1:2] + keys[8:9] + keys[14:15] + keys[23:24]
    arc_test_set_ = arc_test_set.change_keys(keys)
    obj_ds = arc_test_set_.make_object_dataset(make=False)
    bg_cut_ds = obj_ds.make_background_cut_dataset()
    for n,key in enumerate(keys):
        got_length = arc_test_set_.get_length(key,formatter=None,name='input')
        visualize_task(key,title=f"#{n} len input:{got_length}",file='eval')
        # print([v for k,v in obj_ds.background_values.items() if k[0]==key])
        # print([v for k,v in bg_cut_ds.background_values.items() if k[0]==key])
        try:
            if key in bg_cut_ds.keys:
                visualize_task(bg_cut_ds.queries[key],title=f"#{n} background_cut {key}",file='eval')
        except:
            print(bg_cut_ds.queries[key])
            print(traceback.format_exc())

# models
base_model, MyFormatter, perm_aug, max_seq_length_train, mask_first = '/kaggle/input/wb55l_nemomini_fulleval/transformers/default/1', ArcFormatter_premix_3 if pp else ArcFormatter_premix_3, 'rnd_all', 2000 if one_ex else 4224, 0

# training & inference
train_epochs = 24#4
multi_gpu_train = False if train_classifier else True
multi_gpu_random_split = True
max_seq_length_infer = 6336# 2000 if one_ex else 8192
prime_on_single_task = False
infer_params = dict(min_prob=0.17, store=infer_temp_storage, use_turbo=True)

# scoring
use_aug_score = use_aug
aug_score_params = dict(tp=True, rot=True, perm=perm_aug, shfl_ex=True, make_unique=False, max_len=max_seq_length_infer)
submission_select_algo = score_full_probmul_3 if use_aug_score else score_all_probsum

def prepare_run(model_path, load_lora=None, train=False, gpu=None, inf=False, **kwargs):
    if gpu is not None:
        os.environ["CUDA_DEVICE_ORDER"   ] = "PCI_BUS_ID"
        os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu)

    if train_classifier:
        target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
    else:
        target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj', 'embed_tokens', 'lm_head']
        
    model, tokenizer, formatter = prepare_model(  # base model configuration
        model=model_path,
        local_files_only=True,
        mode='unsloth_4bit',
        # mode='transformers',
        #shrink_embedding=8000,
        max_seq_length=max_seq_length_train,
        formatter=MyFormatter,
        inf=inf,
        peft=([dict(
            r=64 if train_classifier else 64 if pp else 64,  # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
            target_modules=target_modules,
            lora_alpha=16 if train_classifier else 64 if pp else 16,
            lora_dropout=0,  # Supports any, but = 0 is optimized
            bias="none",  # Supports any, but = "none" is optimized
            use_gradient_checkpointing=True,  # True or "unsloth" for very long context
            random_state=42,
            use_rslora=True,  # We support rank stabilized LoRA
            loftq_config=None,  # And LoftQ
        )] if train or load_lora else []) + ([load_lora] if load_lora else []),
        **kwargs
    )
    # if gpu==0 and train and pp:print(model)
    
    if train and mask_first: formatter.collator_kwargs.update(mask_first_output=mask_first)

    return model, formatter

def prepare_dataset(formatter, train, gpu=None, key=None):
    ds = arc_test_set
    if key:
        # ds.keys = [key]
        ds = ds.change_keys([key])
        
    ds = ds.make_object_dataset(make=False)
    ds = ds.make_background_cut_dataset()
    if train:
        if do_train:
            ds = ds.move_test_to_train(only_when_2=True)
        ds = ds.remove_replies()
        if get_diff:ds = ds.get_diff()
        ds = ds.augment(tp=True, rot=True, perm=perm_aug, n=1 if do_train else 24 if arc_test_set.is_fake else train_epochs, shfl_ex=True, shfl_keys=True, esc=False, keep_max=keep_max)
        # if one_ex or one_rotp:ds = ds.cut_to_query_count(1)
        ds = ds.cut_to_query_count(cut_query+1)
        if cut_query>2:ds = ds.cut_to_query_count(cut_query,p=0.5)
        if one_rotp:ds = ds.one_rotp()
        # if pp:ds = ds.one_color().shuffled()
        # if pp:ds.keys =  ds.keys[:train_epochs*8]
        ds = ds.cut_to_len(formatter=formatter, name='text', max_len=max_seq_length_train, max_new_tokens=0)
        if pp:print('ds.keys in train prepare_dataset', len(ds.keys), ds.keys[:2])
        # if arc_test_set.is_fake: ds = ds.sorted_by_len(formatter=formatter, name='text', reverse=True)
    else:
        # ds = ds.sorted_by_len(formatter=formatter, name='input', max_of_transposed=True)
        ds = ds.split_multi_replies()
        if get_diff:ds = ds.get_diff()
        ds = ds.augment(tp=True, rot=True, n=1, seed=42, perm=perm_aug, shfl_ex=True, esc=False, keep_max=keep_max).interleave(ds.length())
        # if one_ex or one_rotp:ds = ds.cut_to_query_count(0)

        if use_ext:
            even_keys = [k for n, k in enumerate(ds.keys) if n % 2 == 0]
            odd_keys = [k for n, k in enumerate(ds.keys) if n % 2 != 0]
            
            ds_even = ds.change_keys(even_keys)
            ds_odd  = ds.change_keys(odd_keys)
            
            # Apply transformations to even part
            ds_even = ds_even.remove_replies()
            ds_even = ds_even.last_train_ex_for_test()
            ds_even = ds_even.split_multi_replies(train=True)
            
            # Combine the datasets again
            ds = ArcDataset.append(ds_even, ds_odd)

        ds = ds.cut_to_query_count(cut_query)
        if cut_query>2:ds = ds.cut_to_query_count(cut_query-1,p=0.5)
        if one_rotp:ds = ds.one_rotp()
        # if pp:ds = ds.one_color().shuffled()
        ds = ds.cut_to_len(formatter=formatter, name='input', max_len=max_seq_length_infer)

        # ds = ds.sorted_by_len(reverse=True, formatter=formatter, name='input')
        if pp:print('ds.keys in inference prepare_dataset',len(ds.keys), ds.keys[0:4],ds.keys[-4:])#,'\n-> 128')
        # if arc_test_set.is_fake: ds.keys = ds.keys[:4] #ds.keys[::-1][::5][::-1]

    return ds

def start_training(gpu):
    import gc
    import time
    import json
    import traceback
    time_limit = time.time() + 10.5*60*60

    stats = []
    for nkey,key in enumerate(keys[gpu::gpu_count]):

        if pp:print('\nnkey',nkey,'gpu',gpu,key)
        if time.time() > time_limit or nkey>0 and do_train:
            break
        try:
            train_start_time = time.time()
            storage_path = "Classifier_train_try65" if do_train else f'{model_temp_storage}_gpu{gpu}'
            if (gpu==0 or multi_gpu_train):# and not os.path.exists(storage_path):
                with RemapCudaOOM():
                    # if pp:mem_info('before prepare_run') #Don't get mem info before prepare_run. Raise error.
                    if nkey==0 or not keep_model:
                        use_lora = True
                        model, formatter = prepare_run(base_model, train=use_lora, gpu=gpu if multi_gpu_train else None)
                        # if use_classifier: model = load_classifier_head(model)
                    if pp:mem_info('after prepare_run')
                    prepare_run_fin = time.time()
                    prepare_run_time = prepare_run_fin-train_start_time
                    if pp:print('prepare_run_time',round(prepare_run_time))
                    dataset = prepare_dataset(formatter, train=True, gpu=gpu if multi_gpu_train else None, key=None if do_train else key)
                    t_dataset = prepare_dataset(formatter, train=False, gpu=gpu, key=key)
                    len_test = len(set([v.split('.')[0] for v in t_dataset.keys]))
                    data_len = arc_test_set.get_length(key=key,formatter=formatter,name='text')
                    if pp:
                        sub_formatter = ArcFormatter_premix_3(tokenizer=formatter.tokenizer)
                        data_len_long = arc_test_set.get_length(key=key,formatter=sub_formatter,name='text')
                    if pp:print('data_len',data_len,'data_len_long',data_len_long,'len_test',len_test)
                    # if nkey==gpu==0 and arc_test_set.is_fake:
                    #     if pp:print(arc_test_set.get(key,formatter))
                    if pp:print('data_time',round(time.time()-prepare_run_fin))
                    skip_train = False
                    if skip_train:
                        pass
                    elif train_classifier or use_classifier:
                        # model, loss_log, total_loss = classifier_train_with_accelerate(model, dataset, formatter.tokenizer, formatter, store=storage_path)
                        model, loss_log, total_loss = classifier_train_loop(model, dataset, formatter.tokenizer, formatter, store=storage_path)
                    else:
                        model, trainer_stats, trainer = training_run(
                            model, formatter, dataset, store=storage_path,
                            max_seq_length=max_seq_length_train,
                            grad_acc_fix=False,
                            train_args=dict(
                                per_device_train_batch_size=4 if pp else 4,
                                gradient_accumulation_steps=2 if pp else 2,
                                #from try13&14,b4a1s48 is faster loss drop and redundant inf outpu than b8a1s24, same tr time 147-156s for id0, 240-243s for id3
                                warmup_steps=4,
                                num_train_epochs=1,
                                #max_steps=20 if arc_test_set.is_fake else -1,
                                max_steps=-1,# if do_train else 24 if arc_test_set.is_fake else 24, #ins 20250329
                                learning_rate=1e-4,
                                embedding_learning_rate=1e-5,
                                logging_steps=4,
                                optim="adamw_8bit",
                                weight_decay=0.01,  # 0.01,
                                lr_scheduler_type='constant' if do_train else 'cosine',  # "linear", "cosine", "constant"
                                seed=42,
                                output_dir=os.path.join(tmp_dir, 'checkpoints'),
                                save_strategy="no",
                                report_to='none',
                            ),
                        )

                    train_mem_usage, _ = mem_info('train')

                    del dataset#, model, formatter
                    b_gc = time.time()
                    gc.collect()
                    train_run_fin = time.time()
                    if pp:print('gc time',round(train_run_fin-b_gc))
                    train_run_time = train_run_fin-prepare_run_fin
                    if pp:print('train_run_time', round(train_run_time))
                
                    inf_data_len = arc_test_set.get_length(key=key,name='text',formatter=formatter)
                    if pp:print('inf_data_len',inf_data_len, 'len_test',len_test)
                    retrainer = None if not prime_on_single_task else Retrainer(
                        n=32,
                        aug_opts=dict(perm=perm_aug, shfl_ex=True),
                        reload_state_dict=get_and_fix_peft_weights(storage_path),
                        formatter=formatter,
                        max_seq_length=max_seq_length_infer,
                        grad_acc_fix=False,
                        train_args=dict(
                            per_device_train_batch_size=2,
                            gradient_accumulation_steps=2,
                            warmup_steps=4,
                            num_train_epochs=1,
                            learning_rate=1e-4,
                            embedding_learning_rate=0,
                            max_steps=20 , #ins 20250329
                            logging_steps=8,
                            optim="adamw_8bit",
                            weight_decay=0.00,  # 0.01,
                            lr_scheduler_type='constant',  # "linear", "cosine",
                            seed=42,
                            output_dir='tmp_output',
                            save_strategy='no',
                            report_to='none',
                        ),
                    )
                    decoder = Decoder(formatter, arc_test_set.keep_key_startswith(key).split_multi_replies(), n_guesses=2, prob_baseline=0.05, keep_max=keep_max)
                    if 1:# gpu in [0,1,2,3] and len_test>0 and not pp or gpu in [0,1,2,3] and pp and not use_classifier:
                        import traceback
                        try:
                            if not keep_model: del model, formatter
                            gc.collect()
                            torch.cuda.empty_cache()

                            if pp:mem_info('before_inf_batch')
                            if pp:time.sleep(5)
                            batch_result = inference_run_batch(t_dataset, storage_path=storage_path, gpu=gpu, max_subprocesses=2 if gpu_count==4 else 1, **{k:v if k=='min_prob' and pp else v for k,v in infer_params.items()})
                            inf_mem_usage = batch_result['mem']
                            if not keep_model: model, formatter = None, None

                        except:
                            print('tried inference_run_batch',traceback.format_exc())
                            if not keep_model: model, formatter = prepare_run(storage_path, gpu=gpu)
                            inference_run_v2(model, formatter, t_dataset, decoder, retrain=retrainer, **infer_params)
                            inf_mem_usage,_ = mem_info('inf')

                    else:
                        inference_run_v2(model, formatter, t_dataset, decoder, classifier=True, retrain=retrainer, **infer_params)
                        inf_mem_usage,_ = mem_info('inf')
                    inf_run_fin = time.time()
                    inf_run_time = inf_run_fin-train_run_fin
                    if pp:print('inf_run_time',round(inf_run_time))
                    if use_aug_score or arc_test_set.is_fake: decoder.calc_augmented_scores(model=model, store=score_temp_storage, quiet=not pp, **aug_score_params)
                    calc_fin = time.time()
                    calc_time = calc_fin-inf_run_fin
                    if pp:print('calc_time',round(calc_time))
                    calc_mem_usage,_ = mem_info('calc')
                    
                    if not keep_model: del model, formatter, t_dataset, decoder
                    if os.path.exists(storage_path) and not do_train:
                        import shutil
                        shutil.rmtree(storage_path)
                    gc.collect()

            total_time = time.time()-train_start_time
            if pp:print('Train Inf nkey',nkey,'gpu',gpu,key, round(total_time),'s')
            if pp:print('  data_len',data_len)
            if pp:print('  prepare_run_time',round(prepare_run_time))
            if pp:print('  train_run_time', round(train_run_time))
            if pp:print('  train_mem_usage', train_mem_usage)
            if pp:print('  inf_data_len',inf_data_len)
            if pp:print('  inf_run_time',round(inf_run_time),'inf_run_time/n',round(inf_run_time/(len_test+0.001)))
            if pp:print('  inf_mem_usage',inf_mem_usage)
            if pp:print('  calc_time',round(calc_time))
            if pp:print('  calc_mem_usage',calc_mem_usage)
            if pp:
                if skip_train:
                    loss_log = {}
                    total_loss = 0
                
                elif not train_classifier and not use_classifier:
                    total_loss = trainer_stats.training_loss
                    loss_log = trainer.state.log_history
                stat = {
                    'nkey': nkey,
                    'gpu': str(gpu),
                    'key': key,
                    'total_time': total_time,
                    'data_len': data_len,
                    'data_len_long': data_len_long,
                    'prepare_run_time': prepare_run_time,
                    'train_run_time': train_run_time,
                    'train_mem_usage': train_mem_usage,
                    'train_loss': total_loss,
                    'loss': {v['step']:v['loss'] for v in loss_log if 'loss' in v},
                    # 'disk_w': disk_w,
                    # 'disk_t': disk_t,
                    'inf_data_len': inf_data_len,
                    'len_test': len_test,
                    'inf_run_time': inf_run_time,
                    'inf_run_time/n': inf_run_time/(len_test+0.001),
                    'inf_mem_usage': inf_mem_usage,
                    'calc_time': calc_time,
                    'calc_mem_usage': calc_mem_usage,
                }
                stats.append(stat)

        except:
            print(traceback.format_exc())
            try:
                if not keep_model:
                    del model
                gc.collect()
                del dataset
                gc.collect()
                del decoder
                gc.collect()
                if not keep_model:
                    del formatter
                gc.collect()
            except:
                pass

    if pp:
        with open(f"stats{gpu}.json","w") as f:
            json.dump(stats,f,indent=2)

        import pandas as pd
        
        # Convert to DataFrame
        df = pd.DataFrame(stats)
        
        # List of time-related fields
        time_fields = [
            'total_time',
            'prepare_run_time',
            'train_run_time',
            'inf_run_time',
            'inf_run_time/n',
            'calc_time'
        ]
        
        # Print total sum for each field
        print("\n=== Total Time Summary ===")
        print('of len',len(stats))
        for field in time_fields:
            if field in df.columns:
                total = df[field].sum()
                print(f"{field} Total: {round(total)} s. Ave:",round(total/len(stats)))
                
        mem_fields = ['train_mem_usage', 'inf_mem_usage', 'calc_mem_usage']
        
        print("\n=== Memory Usage Summary (GB) ===")
        for field in mem_fields:
            if field in df.columns:
                avg = df[field].mean()
                max_val = df[field].max()
                print(f"{field}: avg = {round(avg, 2)} GB, max = {round(max_val, 2)} GB")


class RemapCudaOOM:
    def __enter__(self): pass
    def __exit__(self, exc_type, exc_value, traceback):
        oom_errors = ["CUDA out of memory", "Make sure you have enough GPU RAM", "does not fit any GPU's remaining memory"]
        if exc_value and any(x in str(exc_value) for x in oom_errors):
            with open('submission.json', 'w') as f: f.write('cause submission scoring error')


In [ ]:
from common_stuff import *
import os
import time
time_limit = time.time() + 10*60*60
os.environ["WANDB_DISABLED"] = "true"

if not os.path.exists(os.path.join(tmp_dir, 'unsloth_installed')):  # unsloth offline install - https://stackoverflow.com/a/51646354
    !pip uninstall -q --yes torch accelerate
    !pip install -q --no-index --find-links=/kaggle/input/unsloth-2024-9-post4/wheelhouse unsloth
    #!pip uninstall --yes accelerate fastai torch torchaudio transformers
    #!pip install --no-index --find-links=/kaggle/input/unsloth-2024-10-7/wheelhouse unsloth  # do not use grad_acc_fix - trains very slow
    #!sed -i 's/if ((post_check - pre_check) >= 1).sum() > 1:/if False:/g' /opt/conda/lib/python3.10/site-packages/unsloth/models/llama.py
    # fix delay bug in get_statistics()
    !sed -i 's/^def get_statistics():/def get_statistics():\n if False:/g' /opt/conda/lib/python3.10/site-packages/unsloth/models/_utils.py
    # fix faulty unsloth multi-gpu detection
    !sed -i "s/raise RuntimeError('Unsloth currently does not support multi GPU setups - but we are working on it!')/pass/g" /opt/conda/lib/python3.10/site-packages/unsloth/tokenizer_utils.py /opt/conda/lib/python3.10/site-packages/unsloth/models/llama.py /opt/conda/lib/python3.10/site-packages/unsloth/models/vision.py
    os.makedirs(os.path.join(tmp_dir, 'unsloth_installed'), exist_ok=True)
    print('Unsloth installed & patched.')

# for gpu in [0, 1, 2, 3]: 
#     signal_path = f'{model_temp_storage}_gpu{gpu}_done'
#     if os.path.exists(signal_path): os.rmdir(signal_path)

# if arc_test_set.is_fake:  # cleanup? (for debugging)
#     #!rm -R /kaggle/temp/finetuned_model*
#     #!rm -R /kaggle/temp/inference_outputs
#     #!rm -R /kaggle/temp/inference_scoring
#     #!ls /kaggle/temp
#     pass

In [ ]:
%%python --bg --proc train_proc0
from common_stuff import *
start_training(gpu=0)

In [ ]:
%%python --bg --proc train_proc1
from common_stuff import *
start_training(gpu=1)

In [ ]:
%%python --bg --proc train_proc2
from common_stuff import *
start_training(gpu=2)

In [ ]:
%%python --bg --proc train_proc3
from common_stuff import *
start_training(gpu=3)

In [ ]:
if 0:# do_train:
    # from common_stuff import *
    start_training(0)
else:
    train_procs = [train_proc0, train_proc1, train_proc2, train_proc3][:1 if do_train else gpu_count]
    proc_exit_codes = await wait_for_subprocesses(*train_procs, print_output=True or arc_test_set.is_fake)
    print(f'*** Subprocesses exit codes: {proc_exit_codes}')
    assert all(x==0 for x in proc_exit_codes)
if pp:
    import subprocess
    
    result = subprocess.run("du -sh * | sort -hr", shell=True, capture_output=True, text=True)
    print(result.stdout)

    mem_info()

In [ ]:
# write submission
from common_stuff import *
import time, traceback
start_timer = time.time()
with RemapCudaOOM():
    model, formatter, dataset = None, MyFormatter(), None
    decoder = Decoder(formatter, arc_test_set.split_multi_replies(), n_guesses=2, frac_score=True, keep_max=keep_max).from_store(infer_params['store'])
    if use_aug_score or arc_test_set.is_fake: decoder.calc_augmented_scores(model=model, store=score_temp_storage, quiet=not pp, **aug_score_params)
    print('Time calc_augmented_scores',round(time.time()-start_timer))
    submission = arc_test_set.get_submission(decoder.run_selection_algo(submission_select_algo))
    with open('submission.json', 'w') as f: json.dump(submission, f)
    print('Time get_submission',round(time.time()-start_timer))
    if arc_test_set.is_fake:
        try:
            decoder.benchmark_selection_algos(selection_algorithms)
            with open('submission.json') as f: reload_submission = json.load(f)
            print('*** Reload score:', arc_test_set.validate_submission(reload_submission,arc_challenge_file))
        except:
            print(traceback.format_exc())
if pp:
    import subprocess
    
    result = subprocess.run("du -sh * | sort -hr", shell=True, capture_output=True, text=True)
    print(result.stdout)

    mem_info()

In [ ]:
if arc_test_set.is_fake and pp:
    try:
        import pandas as pd
        import plotly.express as px
        import json
        import traceback
        
        stats = []
        for i in range(gpu_count):
            with open(f"stats{i}.json", "r") as f:
                stats.extend(json.load(f))
        
        # Convert to DataFrame
        df = pd.DataFrame(stats)
        display(df)
        
        # Melt the DataFrame to long format
        df_long = df.melt(id_vars='key', value_vars=['data_len', 'data_len_long'], var_name='Type', value_name='Value')
        
        # Grouped Bar Chart
        fig = px.bar(df_long, x='key', y='Value', color='Type', barmode='group', title="data_len by Key")
        fig.show()
        
        fig = px.bar(df, x="key", y="prepare_run_time", color="gpu", title="Prepare Run Time by Key")
        fig.show()
        
        fig = px.bar(df, x="key", y="total_time", color="nkey", title="total_time by Key")
        fig.show()
        
        fig = px.bar(df, x="key", y="train_run_time", color="nkey", title="train_run_time by Key")
        fig.show()
        
        fig = px.bar(df, x="key", y="inf_run_time/n", color="nkey", title="inf_run_time/n by Key")
        fig.show()
        
        fig = px.bar(df, x="key", y="calc_time", color="nkey", title="calc_time by Key")
        fig.show()
        
        # fig = px.scatter(df, x="nkey", y="disk_w", color="gpu", title="disk_w")
        # fig.show()
        
        # fig = px.scatter(df, x="nkey", y="disk_t", color="gpu", title="disk_t")
        # fig.show()
        
        fig = px.scatter(df, x="data_len", y="total_time", color="gpu", title="Total Time vs Data Length")
        fig.show()
        
        fig = px.scatter(df, x="inf_data_len", y="total_time", color="gpu", title="Total Time vs Inference Data Length")
        fig.show()
        
        fig = px.scatter(df, x="data_len", y="train_run_time", color="gpu", title="Train Time vs Data Length")
        fig.show()
        
        fig = px.scatter(df, x="inf_data_len", y="inf_run_time", color="gpu", title="Inference Time vs Inference Data Length")
        fig.show()
        
        fig = px.scatter(df, x="data_len", y="train_mem_usage", color="gpu", title="Train Memory Usage vs Data Length")
        fig.show()
        
        fig = px.scatter(df, x="inf_data_len", y="inf_mem_usage", color="gpu", title="Inference Memory Usage vs Inference Data Length")
        fig.show()
        
        fig = px.scatter(df, x="data_len", y="calc_time", color="gpu", title="Calc Time vs Data Length")
        fig.show()
        
        fig = px.bar(df, x="key", y="train_loss", color="train_run_time", title="train_loss by Key")
        fig.show()

        loss_rows = []
        for row in stats:
            key = row["key"]
            loss_dict = row["loss"]
            for step, loss in loss_dict.items():
                loss_rows.append({"key": key, "step": int(step), "loss": loss})
        
        loss_df = pd.DataFrame(loss_rows)
        loss_df = loss_df.sort_values(by=["key", "step"])  # optional: sort for prettier plot
        
        fig = px.line(loss_df, x="step", y="loss", color="key", markers=True,
                      title="Loss over Steps by Key")
        fig.update_layout(xaxis_title="Step", yaxis_title="Loss")
        fig.show()
        
        fig = px.line(loss_df, x="step", y="loss", color="key", markers=True,
                      title="Loss over Steps by Key", log_y=True)
        fig.update_layout(xaxis_title="Step", yaxis_title="Loss")
        fig.show()
    except:
        print(traceback.format_exc())
        pass
